# Changes From the 32.22 LB Notebook

This notebook keeps the original 32.22 LB solution structure, but adds the following stability and selection fixes.

## 1. Deterministic Per-Puzzle Seeds

The old 32.22 notebook used Python `hash(key)` for puzzle seeds.  
Because Python hash values can change between runs, the same notebook could produce noticeably different LB scores.

This notebook replaces that behavior with deterministic per-puzzle seeding, so each puzzle gets a stable seed across reruns.

## 2. Deterministic Augmentation Seeds

Augmentation seeding is also made deterministic.

This reduces rerun noise and makes debug comparisons more reliable, because the same puzzle should receive the same augmentation behavior across runs.

## 3. Queue Race Condition Fix

The worker queue logic is fixed so an empty queue does not prematurely stop processing.

This prevents puzzles from being left unprocessed during multiprocessing debug or full runs.

## 4. Worker Sentinel Count Fix

The number of sentinel values now matches the number of worker processes.

This avoids incomplete multiprocessing shutdown behavior and reduces the risk of missing outputs.

## 5. Rerun Mode Protection

The notebook has protection against accidentally running only hardcoded debug puzzles during a real Kaggle rerun.

Debug mode can still use `DEBUG_PUZZLES`, but private/test rerun mode should process the full available puzzle set.

## 6. Top-3 Candidate-Pool Dual Selection

The old selection behavior was closer to rank-1 candidate selection.

This notebook uses a top-3 candidate pool from the main scoring signals for dual-answer selection.  
The first attempt remains the strongest candidate, while the second attempt is chosen from the best alternative candidates.

## 7. Removed Unstable Frequency Weighting

Frequency/stability weighting is not used here.

That idea was tested separately and hurt LB performance, so this notebook keeps the cleaner candidate-pool selection logic.

## Current Role of This Notebook

This notebook is treated as the stable base after the 32.22 LB notebook.

In [1]:
import time
# Let starter.py compute 12 hours from its own start
global_end_time = 0.0

In [2]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [3]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    if not (isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)):
        return False
    if np.any((guess < 0) | (guess > 9)):
        return False
    return True
    
def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Invalid permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            # Reject empty or jagged grids
            if not by_rows:
                return None
            row_len = len(by_rows[0])
            if not all(len(r) == row_len for r in by_rows):
                return None
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception as e:
            print(f"[DEBUG] convert_tokens_to_array failed: {type(e).__name__}: {e}")
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))
    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not query['train']:
                    break
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            num_guesses = len(v)
            for i in range(len(target_dict)):
                if i < num_guesses:
                    target_dict[f'attempt_{i+1}'] = v[i].tolist()
                else:
                    target_dict[f'attempt_{i+1}'] = v[-1].tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [4]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs


def getter_full_probmul_3(guesses, baseline=None):
    # Dynamic baseline to avoid negative scores (from V1)
    if baseline is None:
        beam_scores = [g["beam_score"] for g in guesses]
        if not beam_scores:
            return 0
        baseline = max(beam_scores) + 1.0
    inf_score = np.sum([baseline - g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline - s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_dual_selection(self):
        """Return {base_key: [best_from_probmul, best_alternative_from_pool]}.
        
        Uses top-3 candidates from both scoring algorithms to create a candidate pool.
        Attempt 1: score_full_probmul_3's top pick (unchanged, safest).
        Attempt 2: best unique candidate from pool that is not attempt 1.
        This catches rank-2/3 correct solutions (proven: 7b0280bc, 2d0172a1)
        without degrading performance on other puzzles.
        """
        results = {}
        for bk, guesses in self.decoded_results.items():
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            ranked_b = score_kgmon(g)

            # Create a pool of unique candidates from top-3 of both rankers
            candidates_pool = []
            seen = set()
            for cand in ranked_a[:3] + ranked_b[:3]:
                h = hashable(cand)
                if h not in seen:
                    seen.add(h)
                    candidates_pool.append(cand)

            # Attempt 1: best from probmul (keeps original logic)
            top_a = candidates_pool[0] if candidates_pool else None
            # Attempt 2: best unique alternative from the expanded pool
            top_b = candidates_pool[1] if len(candidates_pool) > 1 else top_a

            combined = [x for x in [top_a, top_b] if x is not None]
            results[bk] = combined
        return results
        
    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.rsplit("_", 1)
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg correct beam score: N/A (no correct solutions found)")
            print(" max correct beam score: N/A (no correct solutions found)")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.rsplit("_", 1)[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

        # Dual selection benchmark (this is what our actual submission uses)
        dual_selected = self.run_dual_selection()
        correct_puzzles_dual = {
            k for k, v in dual_selected.items()
            if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])
        }
        print(correct_puzzles_dual)
        score_dual = sum(1/num_tasks_per_puzzle[k.rsplit("_", 1)[0]] for k in correct_puzzles_dual)
        print(f" acc: {score_dual:5.1f}/{num_puzzles:3} ('run_dual_selection — ACTUAL SUBMISSION METHOD')")

Writing arc_decoder.py


In [5]:
%%writefile arc_solver.py

import os
os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import copy
import queue
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

import hashlib

def deterministic_seed(s, mod=10000):
    """
    Python's built-in hash() is randomized by PYTHONHASHSEED,
    so it gives different values across processes/runs.
    This function provides a consistent, reproducible seed across all runs.
    """
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % mod

logging.disable(logging.WARNING)

# ------------------------------------------------------------
# Fixed token IDs (as used in fine-tuning)
# ------------------------------------------------------------
ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_BEAM_CANDIDATES = 10
class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            # Ensure labels is 1D (avoid scalar tensor)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            # Use torch.nonzero instead of torch.where to avoid bool scalar issues
            user_mask = (labels == USER_TOKEN_ID)
            assistant_mask = (labels == ASSISTANT_TOKEN_ID)
            user_start_idx = torch.nonzero(user_mask, as_tuple=True)[0].tolist()
            assistant_start_idx = torch.nonzero(assistant_mask, as_tuple=True)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            eos_mask = (labels == EOS_ID)
            end_idx = torch.nonzero(eos_mask, as_tuple=True)[0].tolist()
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                if start >= end:
                    continue
                if j % 2 == 1:  # assistant response part
                    start += 2
                    end += 1
                    if start < end and end <= labels.size(0):
                        batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        # Direct cache pass (fast, no clone)
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,
            start_time=start_time,
            end_time=end_time,
        )

        del outputs
        # torch.cuda.empty_cache() removed for speed

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    
    max_len = max(batch_lengths)
    padded_tokens = []
    attention_masks = []
    for tokens in batch_tokens:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        padded_tokens.append(padded)
        attention_masks.append([1] * len(tokens) + [0] * pad_len)
        
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(attention_masks, device=model.device, dtype=torch.long)
    
    # FIX: use_cache=False because padding breaks KV cache
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    
    result = []
    # FIX: Slicing on GPU before transferring to CPU to prevent massive OOM Crash
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_logits = outputs.logits[i, query_length-1 : query_length-1+len(answer_tokens), :]
        answer_log_probs = answer_logits.float().log_softmax(-1).cpu()
        answer_score = answer_log_probs[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
        
    return result


def worker(rank, p_queue, end_time):
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch_fused",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    def save_fallback_result(bk, reason):
        try:
            puzzle_id, test_id = bk.rsplit("_", 1)
            test_input = np.asarray(
                arc_test_set.queries[puzzle_id]["test"][int(test_id)]["input"],
                dtype=int,
            )
            fallback_grid = np.zeros_like(test_input, dtype=int)
        except Exception:
            fallback_grid = np.zeros((1, 1), dtype=int)

        fallback_result = [{
            "beam_score": 999.0,
            "score_aug": [999.0] * 8,
            "solution": fallback_grid,
        }]

        fallback_path = os.path.join(dir_outputs, f"{bk}.fallback")
        with bz2.BZ2File(fallback_path, "w") as f:
            pickle.dump(fallback_result, f)

        print(f"[Rank {rank}] FALLBACK_SAVE {bk}: {reason}", flush=True)

    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!", flush=True)
            break

        try:
            key = p_queue.get(timeout=60)
        except queue.Empty:
            continue

        if key is None:
            break

        start_time = time.time()
        processed_bks = set()

        try:
            torch.cuda.reset_peak_memory_stats()

            set_peft_model_state_dict(model, default_weights.copy(), adapter_name="default")
            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])

            train_seed = deterministic_seed(key)
            print(f"[Rank {rank}] train_seed={train_seed} for puzzle {key}", flush=True)
            train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=train_seed)
            train_ds = train_ds.cut_to_len(
                formatter=formatter,
                name="text",
                max_len=max_seq_length,
            )

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )
                stats = trainer.train()
                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                del trainer

            model = FastLanguageModel.for_inference(model)

            gc.collect()
            torch.cuda.empty_cache()

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training", flush=True)
            torch.cuda.reset_peak_memory_stats()
            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}", flush=True)

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_seed = deterministic_seed(f"{key}_eval")
            print(f"[Rank {rank}] eval_seed={eval_seed} for puzzle {key}", flush=True)
            eval_ds = puzzle_ds_multi.augment(n=2, seed=eval_seed)
            eval_ds = eval_ds.cut_to_len(
                formatter=formatter,
                name="input",
                max_len=max_seq_length - max_new_tokens,
            )

            test_id_to_subkeys = defaultdict(list)
            for subkey in sorted(eval_ds.keys):
                test_id = subkey.split(".")[0].split("_")[1]
                test_id_to_subkeys[test_id].append(subkey)

            batches = []
            for test_id, subkeys in test_id_to_subkeys.items():
                batch = []
                for offset in [0, 4]:
                    batch.extend(subkeys[offset:offset + 2])
                if batch:
                    batches.append(batch)

                batch = []
                for offset in [2, 6]:
                    batch.extend(subkeys[offset:offset + 2])
                if batch:
                    batches.append(batch)

            for test_id, subkeys in test_id_to_subkeys.items():
                batch = []
                for offset in [8, 12]:
                    batch.extend(subkeys[offset:offset + 2])
                if batch:
                    batches.append(batch)

                batch = []
                for offset in [10, 14]:
                    batch.extend(subkeys[offset:offset + 2])
                if batch:
                    batches.append(batch)

            with torch.inference_mode():
                known_scores = {}

                for subkeys in batches:
                    spend_time = time.time() - start_time
                    if spend_time > 1600 or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}", flush=True)
                        break

                    print(f"[Rank {rank}] decoding {subkeys}", flush=True)

                    token_groups = defaultdict(list)
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        input_tokens = tokenizer.encode(data["input"])
                        token_groups[len(input_tokens)].append((subkey, input_tokens))

                    if len(token_groups) > 1:
                        print(
                            f"[Rank {rank}] split uneven token lengths "
                            f"{ {k: len(v) for k, v in token_groups.items()} }",
                            flush=True,
                        )

                    for token_len, group_items in sorted(token_groups.items()):
                        group_subkeys = [x[0] for x in group_items]
                        group_tokens = [x[1] for x in group_items]

                        try:
                            dfs_result = inference_turbo_dfs(
                                model, group_tokens, max_new_tokens, max_score, end_time
                            )
                        except Exception as e:
                            print(
                                f"[Rank {rank}] DFS_EXCEPTION token_len={token_len} "
                                f"subkeys={group_subkeys}: {type(e).__name__}: {e}",
                                flush=True,
                            )
                            for subkey in group_subkeys:
                                bk = subkey.split(".")[0]
                                processed_bks.add(bk)
                                save_fallback_result(bk, f"DFS_EXCEPTION {type(e).__name__}")
                            continue

                        for subkey_id, scored_beams in dfs_result:
                            subkey = group_subkeys[subkey_id]
                            bk = subkey.split(".")[0]
                            processed_bks.add(bk)
                            decoded_result = []

                            for beam_score, beam_tokens in scored_beams:
                                try:
                                    array = formatter.convert_tokens_to_array(beam_tokens)
                                    if array is None:
                                        continue

                                    solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                                except Exception as e:
                                    print(
                                        f"[Rank {rank}] DECODE_EXCEPTION {subkey}: "
                                        f"{type(e).__name__}: {e}",
                                        flush=True,
                                    )
                                    continue

                                grid_id = (bk, tuple(map(tuple, solution)))

                                if grid_id in known_scores:
                                    augmented_scores = known_scores[grid_id]
                                else:
                                    print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}", flush=True)

                                    try:
                                        aug_dataset = ArcDataset(
                                            keys=[bk],
                                            queries={bk: puzzle_ds_multi.queries.get(bk)},
                                            replies={bk: [solution.tolist()]},
                                        )
                                        aug_seed = deterministic_seed(f"{bk}_aug", mod=1024**2)
                                        aug_dataset = aug_dataset.augment(seed=aug_seed)
                                        aug_dataset = aug_dataset.cut_to_len(
                                            formatter=formatter,
                                            name="input",
                                            max_len=max_seq_length - max_new_tokens,
                                        )

                                        aug_queries = []
                                        aug_answers = []
                                        for augmented_sample in aug_dataset.as_list(formatter):
                                            aug_queries.append(augmented_sample["input"])
                                            aug_answers.append(augmented_sample["reply"])

                                        augmented_scores1 = calc_scores(
                                            aug_queries[:4], aug_answers[:4], tokenizer, model
                                        )
                                        augmented_scores2 = calc_scores(
                                            aug_queries[4:], aug_answers[4:], tokenizer, model
                                        )
                                        augmented_scores = augmented_scores1 + augmented_scores2
                                    except Exception as e:
                                        print(
                                            f"[Rank {rank}] SCORE_EXCEPTION {subkey}: "
                                            f"{type(e).__name__}: {e}",
                                            flush=True,
                                        )
                                        augmented_scores = [999.0] * 8

                                    known_scores[grid_id] = augmented_scores

                                decoded_result.append({
                                    "beam_score": beam_score,
                                    "score_aug": augmented_scores,
                                    "solution": solution,
                                })

                            if decoded_result:
                                with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                    pickle.dump(decoded_result, f)
                            else:
                                save_fallback_result(bk, "EMPTY_DECODED_RESULT")

            for i in range(len(arc_test_set.queries[key]["test"])):
                bk = f"{key}_{i}"
                if bk not in processed_bks:
                    save_fallback_result(bk, "NOT_PROCESSED")

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference", flush=True)
            spend_time = time.time() - start_time
            print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s", flush=True)

        except Exception as e:
            print(
                f"[Rank {rank}] PUZZLE_EXCEPTION {key}: {type(e).__name__}: {e}",
                flush=True,
            )

            try:
                for i in range(len(arc_test_set.queries[key]["test"])):
                    save_fallback_result(f"{key}_{i}", f"PUZZLE_EXCEPTION {type(e).__name__}")
            except Exception as fallback_error:
                print(
                    f"[Rank {rank}] FALLBACK_FAILED {key}: "
                    f"{type(fallback_error).__name__}: {fallback_error}",
                    flush=True,
                )

            gc.collect()
            torch.cuda.empty_cache()
            continue

Writing arc_solver.py


In [6]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import traceback
import numpy as np
import torch.multiprocessing as mp

# ──────────────────────────────────────────────────────────────
#   Debug mode: only these puzzles run during manual execution.
#   In Kaggle rerun mode, all puzzles are processed automatically.
#   All IDs are from the EVALUATION set (arc-agi_evaluation_challenges.json)
#   to ensure they are found during manual runs.
# ──────────────────────────────────────────────────────────────
RUN_FULL_EVAL = True
NPROCS = 4  # number of worker processes

# DEBUG_PUZZLES = [
#     "0934a4d8",  # tier=3, fine=1.7619, tags=tier3,dynamic,rare,high_ratio,extreme_ratio,many_colors,large,dense_non_bg
#     "20a9e565",  # tier=3, fine=1.5125, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,many_colors,large,sparse_bg
#     "4c7dc4dd",  # tier=3, fine=1.8181, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,large,dense_non_bg,low_train
#     "89565ca0",  # tier=3, fine=1.4844, tags=tier3,dynamic,high_ratio,extreme_ratio,many_colors,large
#     "136b0064",  # tier=1, fine=0.4736, tags=tier1,hard_tier1,dynamic,rare,sparse_bg
#     "291dc1e1",  # tier=1, fine=0.5169, tags=tier1,hard_tier1,dynamic,rare,many_colors,dense_non_bg
#     "16b78196",  # tier=2, fine=0.4400, tags=tier2,rare,large,sparse_bg,low_train
#     "271d71e2",  # tier=1, fine=0.1552, tags=tier1,easy_static,easy,rare,dense_non_bg
#     "28a6681f",  # tier=1, fine=0.1733, tags=tier1,easy,rare
#     "2b83f449",  # tier=1, fine=0.1705, tags=tier1,easy_static,easy,rare,low_train
#     "2c181942",  # tier=2, fine=0.2887, tags=tier2,rare,dense_non_bg
#     "31f7f899",  # tier=1, fine=0.2092, tags=tier1,easy,rare,many_colors,dense_non_bg
#     "35ab12c3",  # tier=1, fine=0.2654, tags=tier1,rare,many_colors,sparse_bg
#     "3dc255db",  # tier=1, fine=0.1631, tags=tier1,easy,rare,sparse_bg
#     "409aa875",  # tier=1, fine=0.1853, tags=tier1,easy_static,easy,rare,dense_non_bg
#     "4c416de3",  # tier=2, fine=0.3368, tags=tier2,rare,many_colors,dense_non_bg
#     "64efde09",  # tier=2, fine=0.3802, tags=tier2,rare,many_colors,large,dense_non_bg,low_train
#     "6e453dd6",  # tier=1, fine=0.1484, tags=tier1,easy_static,easy,rare,dense_non_bg
#     "7491f3cf",  # tier=1, fine=0.2583, tags=tier1,rare,many_colors,dense_non_bg
#     "7b0280bc",  # tier=1, fine=0.3054, tags=tier1,rare,many_colors,dense_non_bg
#     "7b80bb43",  # tier=2, fine=0.2720, tags=tier2,rare,large,low_train
#     "7c66cb00",  # tier=2, fine=0.3800, tags=tier2,rare,large,dense_non_bg
#     "1818057f",  # tier=1, fine=0.1177, tags=tier1,easy_static,easy,dense_non_bg
#     "332f06d7",  # tier=1, fine=0.1380, tags=tier1,easy_static,easy,dense_non_bg
#     "53fb4810",  # tier=1, fine=0.2058, tags=tier1,easy_static,easy,dense_non_bg,low_train
#     "7666fa5d",  # tier=1, fine=0.1653, tags=tier1,easy_static,easy,dense_non_bg,low_train
#     "da515329",  # tier=1, fine=0.1382, tags=tier1,easy_static,easy,sparse_bg
#     "8f3a5a89",  # tier=1, fine=0.1640, tags=tier1,easy_static,easy,rare,dense_non_bg
#     "db695cfb",  # tier=1, fine=0.1465, tags=tier1,easy_static,easy,rare,dense_non_bg
#     "97d7923e",  # tier=1, fine=0.1644, tags=tier1,easy,sparse_bg
#     "de809cff",  # tier=1, fine=0.2533, tags=tier1,dense_non_bg,low_train
#     "c7f57c3e",  # tier=1, fine=0.2280, tags=tier1,rare,dense_non_bg,low_train
#     "cb2d8a2c",  # tier=1, fine=0.1766, tags=tier1,easy_static,easy,multi_query,dense_non_bg
#     "58490d8a",  # tier=2, fine=0.7666, tags=tier2,dynamic,high_ratio,dense_non_bg
#     "dbff022c",  # tier=1, fine=0.2338, tags=tier1,rare,many_colors
#     "5961cc34",  # tier=2, fine=0.2485, tags=tier2,dense_non_bg
#     "80a900e0",  # tier=2, fine=0.2867, tags=tier2,large,low_train
#     "269e22fb",  # tier=1, fine=0.4397, tags=tier1,dynamic,multi_query,dense_non_bg
#     "8b9c3697",  # tier=2, fine=0.2593, tags=tier2,dense_non_bg
#     "e376de54",  # tier=1, fine=0.2253, tags=tier1,rare,dense_non_bg
#     "b99e7126",  # tier=2, fine=0.3803, tags=tier2,large,dense_non_bg
#     "c4d067a0",  # tier=2, fine=0.2763, tags=tier2,large,dense_non_bg
#     "135a2760",  # tier=1, fine=0.2078, tags=tier1,easy,rare,dense_non_bg,low_train
#     "4a21e3da",  # tier=1, fine=0.1680, tags=tier1,easy_static,easy,multi_query,dense_non_bg
#     "71e489b6",  # tier=1, fine=0.1528, tags=tier1,easy_static,easy,multi_query
#     "800d221b",  # tier=1, fine=0.2840, tags=tier1,rare,many_colors,dense_non_bg
#     "dd6b8c4b",  # tier=1, fine=0.1403, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
#     "36a08778",  # tier=1, fine=0.0788, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
# ]
# DEBUG_PUZZLES = [
#     # Same-rank selection candidates
#     "142ca369",  # rank 7 / 7
#     "3dc255db",  # rank 5 / 5
#     "4c7dc4dd",  # rank 3 / 3
#     "800d221b",  # rank 4 / 4
#     "dfadab01",  # rank 3 / 3

#     # Control cases (different ranks)
#     "62593bfd",  # rank 6 / 8
#     "8b7bacbf",  # rank 9 / 5

#     # # Already rescued by dual selection
#     # "20270e3b",
#     # "2d0172a1",
#     # "2ba387bc",
#     # "31f7f899",
#     ]


def get_puzzle_priority(key, data):
    train = data[key]['train']
    n_train = len(train)

    avg_input_size = np.mean([
        np.prod(np.shape(x['input'])) for x in train
    ])
    size_ratios = [
        max(np.prod(np.shape(x['output'])),
            np.prod(np.shape(x['input']))) /
        max(min(np.prod(np.shape(x['output'])),
                np.prod(np.shape(x['input']))), 1)
        for x in train
    ]
    max_ratio = max(size_ratios)
    avg_ratio = np.mean(size_ratios)

    n_colors = len(set(
        int(c)
        for x in train
        for arr in [x['input'], x['output']]
        for c in np.unique(arr).tolist()
    ))
    is_dynamic = any(
        np.shape(x['input']) != np.shape(x['output'])
        for x in train
    )

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )

    if n_train >= 6 and tier < 3:
        fine *= 0.7

    return (tier, fine)


def local_worker(rank, queue, end_time, import_lock):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    try:
        # Serialize Unsloth import/patching. This avoids multi-process import race.
        with import_lock:
            print(f"[Rank {rank}] importing arc_solver...", flush=True)
            from arc_solver import worker
            print(f"[Rank {rank}] arc_solver import ok", flush=True)

        print(f"[Rank {rank}] start!", flush=True)
        worker(rank, queue, end_time)
        print(f"[Rank {rank}] done!", flush=True)

    except Exception as e:
        print(f"[Rank {rank}] WORKER_EXCEPTION {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()
        return


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--end-time",
        type=float,
        default=0.0,
        help="Unix timestamp when execution must stop. If 0, compute 12 hours from now.",
    )
    args = parser.parse_args()

    if args.end_time == 0.0:
        end_time = time.time() + 12 * 3600
    else:
        end_time = args.end_time

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    manager = mp.Manager()
    queue = manager.Queue()
    import_lock = manager.Lock()

    keys_sorted = sorted(
        data.keys(),
        key=lambda k: get_puzzle_priority(k, data),
    )

    tiers = [get_puzzle_priority(k, data)[0] for k in keys_sorted]
    print(
        f"[Starter] Puzzle tiers - "
        f"Easy: {tiers.count(1)}, "
        f"Hard: {tiers.count(2)}, "
        f"Extreme: {tiers.count(3)}",
        flush=True,
    )
    print(
        f"[Starter] First: {keys_sorted[0]}, Last: {keys_sorted[-1]}",
        flush=True,
    )

    selected_keys = []

    for key in keys_sorted:
        if not rerun_mode and not RUN_FULL_EVAL:
            if key not in DEBUG_PUZZLES:
                continue
        selected_keys.append(key)
        queue.put(key)

    print(
        f"[Starter] Queued {len(selected_keys)} puzzles "
        f"(rerun_mode={bool(rerun_mode)}, RUN_FULL_EVAL={RUN_FULL_EVAL})",
        flush=True,
    )

    for _ in range(NPROCS):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, end_time, import_lock), nprocs=NPROCS)

Writing starter.py


In [7]:
!UNSLOTH_DISABLE_RL_PATCH=1 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

[Starter] Puzzle tiers - Easy: 65, Hard: 51, Extreme: 4
[Starter] First: 36a08778, Last: 4c7dc4dd
[Starter] Queued 120 puzzles (rerun_mode=False, RUN_FULL_EVAL=True)
[Rank 1] importing arc_solver...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 1] arc_solver import ok
[Rank 1] start!
[Rank 2] importing arc_solver...
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading checkpoint shards:  50%|█████████         | 1/2 [00:32<00:32, 32.20s/it]🦥 Unsloth Zoo will now patch everything to make t

In [8]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_dual_selection())

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

*** Load solutions from '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json'...
*** Generating submission for 172 outputs...
*** Benchmark selection algorithms...
ALL_CORRECT: 1.14514 -  1.40080 30x22 [d8e07eb2_1.rot90.rot90.permute4871623509.ex0312.out1]
ALL_CORRECT: 0.14626 -  1.40080 30x22 [d8e07eb2_1.rot90.permute3145782096.ex0314.out0]
ALL_CORRECT: 1.02354 -  1.40080 30x22 [d8e07eb2_1.rot90.rot90.rot90.permute6807521493.ex4312.out1]
ALL_CORRECT: 0.03054 -  1.40080 30x22 [d8e07eb2_1.transpose.rot90.rot90.rot90.permute4258739061.ex2013.out0]
ALL_CORRECT: 0.60211 -  1.40080 30x22 [d8e07eb2_1.permute7358129604.ex2043.out0]
ALL_CORRECT: 0.12806 -  1.40080 30x22 [d8e07eb2_1.transpose.rot90.rot90.rot90.permute4516970283.ex3240.out0]
ALL_CORRECT: 0.31473 -  1.40080 30x22 [d8e07eb2_1.rot90.permute1398765042.ex2341.out0]
ALL_CORRECT: 0.95170 -  1.40080 30x22 [d8e07eb2_1.rot90.rot90.rot90.permute6750283194.ex4230.out0]
ALL_CORRECT: 0.51507 -  1.40080 30x22 

In [9]:
if decoder.decoded_results:
    test_key = list(decoder.decoded_results.keys())[0]
    result = decoder.run_dual_selection()
    print(f"Key: {test_key}")
    print(f"Attempt 1 shape: {result[test_key][0].shape if result[test_key] else None}")
    print(f"Attempt 2 shape: {result[test_key][1].shape if len(result[test_key]) > 1 else None}")
else:
    print("No results were generated. Model inference likely failed.")

Key: b99e7126_0
Attempt 1 shape: (29, 29)
Attempt 2 shape: (29, 29)


In [10]:
# ================================================================
# DEBUG PUZZLES - MAIN_DEBUG_48_EVAL_PROXY DIAGNOSTIC
# ================================================================
import hashlib
import re
import numpy as np
from starter import get_puzzle_priority
from arc_decoder import score_full_probmul_3, score_kgmon

STATUS_SOLVED = "SOLVED"
STATUS_UNSOLVED = "UNSOLVED"
STATUS_NO_OUTPUT = "NO OUTPUT"

def deterministic_seed(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % 10000

def status_text(row):
    return str(row.get("status", "")).upper()

def is_no_output(row):
    return status_text(row) == STATUS_NO_OUTPUT

def is_unsolved(row):
    return status_text(row) == STATUS_UNSOLVED

def is_solved(row):
    return status_text(row) == STATUS_SOLVED

def dual_yes(row):
    return "YES" in str(row.get("solved_by_dual", "")).upper()

if hasattr(data, "queries"):
    data_dict = data.queries
else:
    data_dict = data


# DEBUG_PUZZLES = [
#     '136b0064', '291dc1e1', '271d71e2',
    # "0934a4d8",  # tier=3, fine=1.7619, tags=tier3,dynamic,rare,high_ratio,extreme_ratio,many_colors,large,dense_non_bg
    # "20a9e565",  # tier=3, fine=1.5125, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,many_colors,large,sparse_bg
    # "4c7dc4dd",  # tier=3, fine=1.8181, tags=tier3,dynamic,rare,multi_query,high_ratio,extreme_ratio,large,dense_non_bg,low_train
    # "89565ca0",  # tier=3, fine=1.4844, tags=tier3,dynamic,high_ratio,extreme_ratio,many_colors,large
    # "136b0064",  # tier=1, fine=0.4736, tags=tier1,hard_tier1,dynamic,rare,sparse_bg
    # "291dc1e1",  # tier=1, fine=0.5169, tags=tier1,hard_tier1,dynamic,rare,many_colors,dense_non_bg
    # "16b78196",  # tier=2, fine=0.4400, tags=tier2,rare,large,sparse_bg,low_train
    # "271d71e2",  # tier=1, fine=0.1552, tags=tier1,easy_static,easy,rare,dense_non_bg
    # "28a6681f",  # tier=1, fine=0.1733, tags=tier1,easy,rare
    # "2b83f449",  # tier=1, fine=0.1705, tags=tier1,easy_static,easy,rare,low_train
    # "2c181942",  # tier=2, fine=0.2887, tags=tier2,rare,dense_non_bg
    # "31f7f899",  # tier=1, fine=0.2092, tags=tier1,easy,rare,many_colors,dense_non_bg
    # "35ab12c3",  # tier=1, fine=0.2654, tags=tier1,rare,many_colors,sparse_bg
    # "3dc255db",  # tier=1, fine=0.1631, tags=tier1,easy,rare,sparse_bg
    # "409aa875",  # tier=1, fine=0.1853, tags=tier1,easy_static,easy,rare,dense_non_bg
    # "4c416de3",  # tier=2, fine=0.3368, tags=tier2,rare,many_colors,dense_non_bg
    # "64efde09",  # tier=2, fine=0.3802, tags=tier2,rare,many_colors,large,dense_non_bg,low_train
    # "6e453dd6",  # tier=1, fine=0.1484, tags=tier1,easy_static,easy,rare,dense_non_bg
    # "7491f3cf",  # tier=1, fine=0.2583, tags=tier1,rare,many_colors,dense_non_bg
    # "7b0280bc",  # tier=1, fine=0.3054, tags=tier1,rare,many_colors,dense_non_bg
    # "7b80bb43",  # tier=2, fine=0.2720, tags=tier2,rare,large,low_train
    # "7c66cb00",  # tier=2, fine=0.3800, tags=tier2,rare,large,dense_non_bg
    # "1818057f",  # tier=1, fine=0.1177, tags=tier1,easy_static,easy,dense_non_bg
    # "332f06d7",  # tier=1, fine=0.1380, tags=tier1,easy_static,easy,dense_non_bg
    # "53fb4810",  # tier=1, fine=0.2058, tags=tier1,easy_static,easy,dense_non_bg,low_train
    # "7666fa5d",  # tier=1, fine=0.1653, tags=tier1,easy_static,easy,dense_non_bg,low_train
    # "da515329",  # tier=1, fine=0.1382, tags=tier1,easy_static,easy,sparse_bg
    # "8f3a5a89",  # tier=1, fine=0.1640, tags=tier1,easy_static,easy,rare,dense_non_bg
    # "db695cfb",  # tier=1, fine=0.1465, tags=tier1,easy_static,easy,rare,dense_non_bg
    # "97d7923e",  # tier=1, fine=0.1644, tags=tier1,easy,sparse_bg
    # "de809cff",  # tier=1, fine=0.2533, tags=tier1,dense_non_bg,low_train
    # "c7f57c3e",  # tier=1, fine=0.2280, tags=tier1,rare,dense_non_bg,low_train
    # "cb2d8a2c",  # tier=1, fine=0.1766, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    # "58490d8a",  # tier=2, fine=0.7666, tags=tier2,dynamic,high_ratio,dense_non_bg
    # "dbff022c",  # tier=1, fine=0.2338, tags=tier1,rare,many_colors
    # "5961cc34",  # tier=2, fine=0.2485, tags=tier2,dense_non_bg
    # "80a900e0",  # tier=2, fine=0.2867, tags=tier2,large,low_train
    # "269e22fb",  # tier=1, fine=0.4397, tags=tier1,dynamic,multi_query,dense_non_bg
    # "8b9c3697",  # tier=2, fine=0.2593, tags=tier2,dense_non_bg
    # "e376de54",  # tier=1, fine=0.2253, tags=tier1,rare,dense_non_bg
    # "b99e7126",  # tier=2, fine=0.3803, tags=tier2,large,dense_non_bg
    # "c4d067a0",  # tier=2, fine=0.2763, tags=tier2,large,dense_non_bg
    # "135a2760",  # tier=1, fine=0.2078, tags=tier1,easy,rare,dense_non_bg,low_train
    # "4a21e3da",  # tier=1, fine=0.1680, tags=tier1,easy_static,easy,multi_query,dense_non_bg
    # "71e489b6",  # tier=1, fine=0.1528, tags=tier1,easy_static,easy,multi_query
    # "800d221b",  # tier=1, fine=0.2840, tags=tier1,rare,many_colors,dense_non_bg
    # "dd6b8c4b",  # tier=1, fine=0.1403, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
    # "36a08778",  # tier=1, fine=0.0788, tags=tier1,easy_static,easy,rare,multi_query,dense_non_bg
# ]
DEBUG_PUZZLES = list(data_dict.keys())
# DEBUG_PUZZLES = [
#     # Same-rank selection candidates
#     "142ca369",  # rank 7 / 7
#     "3dc255db",  # rank 5 / 5
#     "4c7dc4dd",  # rank 3 / 3
#     "800d221b",  # rank 4 / 4
#     "dfadab01",  # rank 3 / 3

#     # Control cases (different ranks)
#     "62593bfd",  # rank 6 / 8
#     "8b7bacbf",  # rank 9 / 5

#     # # Already rescued by dual selection
#     # "20270e3b",
#     # "2d0172a1",
#     # "2ba387bc",
#     # "31f7f899",
# ]
print("=" * 100)
print("CHECK 1: SEED DETERMINISM")
print("=" * 100)
for k in DEBUG_PUZZLES:
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    print(f"{k}: train_seed={ts}, eval_seed={es}")
print("(These values must stay identical across every run/commit.)")

print("\n" + "=" * 100)
print("CHECK 2: PUZZLE ORDERING / TIER ASSIGNMENT")
print("=" * 100)

tier_counts = {1: 0, 2: 0, 3: 0}
for k in DEBUG_PUZZLES:
    if k in data_dict:
        tier, fine = get_puzzle_priority(k, data_dict)
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
        print(f"{k}: tier={tier}, fine_score={fine:.4f}")
    else:
        print(f"{k}: NOT FOUND in current dataset")

print(
    f"\nTier distribution: "
    f"Tier1={tier_counts.get(1, 0)}, "
    f"Tier2={tier_counts.get(2, 0)}, "
    f"Tier3={tier_counts.get(3, 0)}"
)

print("\n" + "=" * 100)
print("CHECK 3: TRAIN/EVAL SEED SEPARATION")
print("=" * 100)

all_separated = True
for k in DEBUG_PUZZLES:
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    if ts == es:
        all_separated = False
        print(f"BAD {k}: train_seed={ts} == eval_seed={es}")
    else:
        print(f"OK  {k}: train_seed={ts} != eval_seed={es}")

print("\nOK: all train/eval seeds are separated." if all_separated else "\nBAD: some seeds are not separated.")

print("\n" + "=" * 100)
print("CHECK 4: SOLVED / UNSOLVED SUMMARY")
print("=" * 100)

summary_data = []

# Actual submission dual-selection result.
# This keeps diagnostic Dual counts aligned with decoder.run_dual_selection().
actual_dual_selection = decoder.run_dual_selection()

for puzzle_id in DEBUG_PUZZLES:
    base_keys = [k for k in decoder.decoded_results.keys() if k.startswith(puzzle_id)]

    if not base_keys:
        summary_data.append({
            "id": puzzle_id,
            "status": STATUS_NO_OUTPUT,
            "solved_by_dual": "No",
            "rank_probmul": "N/A",
            "rank_kgmon": "N/A",
            "best_beam": "N/A",
            "n_queries": 0,
        })
        continue

    solved_flag = False
    solved_by_dual = False
    best_beam = None
    rank_probmul = None
    rank_kgmon = None
    n_queries = len(base_keys)

    for bk in base_keys:
        guesses = dict(decoder.decoded_results.get(bk, {}))
        correct_solution = decoder.dataset.replies.get(bk, [None])[0]

        if correct_solution is None or not guesses:
            continue

        correct_found = False
        for g in guesses.values():
            if np.array_equal(g["solution"], correct_solution):
                correct_found = True
                if best_beam is None or g["beam_score"] < best_beam:
                    best_beam = g["beam_score"]
                break

        if correct_found:
            solved_flag = True

            ranked_a = score_full_probmul_3(guesses)
            ranked_b = score_kgmon(guesses)

            local_rank_probmul = None
            local_rank_kgmon = None

            for i, cand in enumerate(ranked_a):
                if np.array_equal(cand, correct_solution):
                    local_rank_probmul = i + 1
                    break

            for i, cand in enumerate(ranked_b):
                if np.array_equal(cand, correct_solution):
                    local_rank_kgmon = i + 1
                    break

            if local_rank_probmul is not None:
                rank_probmul = local_rank_probmul if rank_probmul is None else min(rank_probmul, local_rank_probmul)
            if local_rank_kgmon is not None:
                rank_kgmon = local_rank_kgmon if rank_kgmon is None else min(rank_kgmon, local_rank_kgmon)

            dual_candidates = actual_dual_selection.get(bk, [])
            if any(np.array_equal(cand, correct_solution) for cand in dual_candidates):
                solved_by_dual = True

    summary_data.append({
        "id": puzzle_id,
        "status": STATUS_SOLVED if solved_flag else STATUS_UNSOLVED,
        "solved_by_dual": "Yes" if solved_by_dual else "No",
        "rank_probmul": rank_probmul if rank_probmul is not None else "N/A",
        "rank_kgmon": rank_kgmon if rank_kgmon is not None else "N/A",
        "best_beam": f"{best_beam:.5f}" if best_beam is not None else "N/A",
        "n_queries": n_queries,
    })

print(f"\n{'Puzzle ID':<15} {'Status':<12} {'Dual':<8} {'Rank Probmul':<14} {'Rank Kgmon':<12} {'Best Beam':<12} {'Queries':<7}")
print("-" * 90)

for row in summary_data:
    print(
        f"{row['id']:<15} {row['status']:<12} {row['solved_by_dual']:<8} "
        f"{str(row['rank_probmul']):<14} {str(row['rank_kgmon']):<12} "
        f"{row['best_beam']:<12} {row['n_queries']:<7}"
    )

total = len(summary_data)
solved_count = sum(1 for r in summary_data if is_solved(r))
dual_solved = sum(1 for r in summary_data if dual_yes(r))

print("\n" + "=" * 100)
print(f"SUMMARY: {solved_count}/{total} puzzles solved in decoded candidates.")
print(f"         {dual_solved}/{total} puzzles solved by dual selection.")
print("=" * 100)

print("\n" + "=" * 100)
print("CHECK 5: TIMEOUT / MISSING OUTPUT BREAKDOWN")
print("=" * 100)

no_output = [r["id"] for r in summary_data if is_no_output(r)]
unsolved = [r["id"] for r in summary_data if is_unsolved(r)]
solved = [r["id"] for r in summary_data if is_solved(r)]

print(f"No output: {len(no_output)} -> {no_output}")
print(f"Unsolved : {len(unsolved)} -> {unsolved}")
print(f"Solved   : {len(solved)} -> {solved}")
print("NO OUTPUT = infrastructure/timeout/queue issue.")
print("UNSOLVED  = ran, but wrong answer/model issue.")

print("\n" + "=" * 100)
print("CHECK 6: MULTI-QUERY PUZZLE BREAKDOWN")
print("=" * 100)

multi_query = [r for r in summary_data if r["n_queries"] > 1]
if multi_query:
    print(f"Multi-query puzzles found: {len(multi_query)}")
    for r in multi_query:
        print(f"  {r['id']}: {r['n_queries']} queries, status={r['status']}, dual={r['solved_by_dual']}")
else:
    print("No multi-query puzzles in DEBUG_PUZZLES.")

print("\n" + "=" * 100)
print("CHECK 7: TIME vs TIER CONSISTENCY")
print("=" * 100)
print("Paste full run log text into log_text below to enable this check.")

log_text = """
PASTE_FULL_RUN_LOG_HERE
"""

times = {}
for match in re.finditer(r"finished (\w+) in ([\d.]+)s", log_text):
    times[match.group(1)] = float(match.group(2))

if times:
    print(f"Found timing data for {len(times)} puzzles:")
    for k, t in sorted(times.items(), key=lambda x: x[1]):
        if k in data_dict:
            tier, _ = get_puzzle_priority(k, data_dict)
            print(f"  {k}: {t:.1f}s, tier={tier}")
else:
    print("No timing data found.")

print("\n" + "=" * 100)
print("CHECK 8: LENS-WISE SOLVE / NO-OUTPUT BREAKDOWN")
print("=" * 100)

def grid_cells(grid):
    return int(np.prod(np.shape(grid)))

def debug_feature_tags(pid, data_dict):
    puzzle = data_dict[pid]
    train = puzzle["train"]

    input_sizes = [grid_cells(ex["input"]) for ex in train]
    output_sizes = [grid_cells(ex["output"]) for ex in train]
    ratios = [
        max(i, o) / max(min(i, o), 1)
        for i, o in zip(input_sizes, output_sizes)
    ]

    colors = set()
    for ex in train:
        for field in ["input", "output"]:
            arr = np.array(ex[field])
            colors.update(np.unique(arr).tolist())

    tier, fine = get_puzzle_priority(pid, data_dict)
    max_ratio = float(max(ratios))
    n_queries = len(puzzle["test"])
    is_dynamic = any(np.shape(ex["input"]) != np.shape(ex["output"]) for ex in train)
    is_rare = bool(6 in colors or 9 in colors)

    tags = [f"tier{tier}"]
    if tier == 1 and (not is_dynamic) and max_ratio == 1 and len(colors) <= 5:
        tags.append("easy_static")
    if tier == 1 and fine <= 0.22:
        tags.append("easy")
    if tier == 1 and fine > 0.45:
        tags.append("hard_tier1")
    if is_dynamic:
        tags.append("dynamic")
    if is_rare:
        tags.append("rare")
    if n_queries > 1:
        tags.append("multi_query")
    if max_ratio > 5:
        tags.append("high_ratio")
    if max_ratio > 30:
        tags.append("extreme_ratio")

    return {
        "tier": tier,
        "fine": fine,
        "tags": tags,
        "rare": is_rare,
        "dynamic": is_dynamic,
        "multi_query": n_queries > 1,
        "high_ratio": max_ratio > 5,
        "extreme_ratio": max_ratio > 30,
    }

feature_by_id = {
    pid: debug_feature_tags(pid, data_dict)
    for pid in DEBUG_PUZZLES
    if pid in data_dict
}

summary_by_id = {r["id"]: r for r in summary_data}

LENSES = {
    "ALL_48": lambda f: True,
    "TIER1": lambda f: f["tier"] == 1,
    "TIER2": lambda f: f["tier"] == 2,
    "TIER3": lambda f: f["tier"] == 3,
    "EASY_STATIC": lambda f: "easy_static" in f["tags"],
    "RARE": lambda f: f["rare"],
    "MULTI_QUERY": lambda f: f["multi_query"],
    "DYNAMIC": lambda f: f["dynamic"],
    "HIGH_RATIO": lambda f: f["high_ratio"],
    "EXTREME_RATIO": lambda f: f["extreme_ratio"],
}

lens_rows = []

for lens_name, lens_fn in LENSES.items():
    ids = [pid for pid, feat in feature_by_id.items() if lens_fn(feat)]

    solved_ids = [pid for pid in ids if is_solved(summary_by_id[pid])]
    no_output_ids = [pid for pid in ids if is_no_output(summary_by_id[pid])]
    unsolved_ids = [pid for pid in ids if is_unsolved(summary_by_id[pid])]
    dual_ids = [pid for pid in ids if dual_yes(summary_by_id[pid])]

    ran_n = len(ids) - len(no_output_ids)
    solve_rate = len(solved_ids) / ran_n * 100 if ran_n else 0.0
    dual_rate = len(dual_ids) / ran_n * 100 if ran_n else 0.0
    no_output_rate = len(no_output_ids) / len(ids) * 100 if ids else 0.0

    lens_rows.append({
        "lens": lens_name,
        "total": len(ids),
        "ran": ran_n,
        "solved": len(solved_ids),
        "dual": len(dual_ids),
        "unsolved": len(unsolved_ids),
        "no_output": len(no_output_ids),
        "solve_rate": solve_rate,
        "dual_rate": dual_rate,
        "no_output_rate": no_output_rate,
        "solved_ids": solved_ids,
        "unsolved_ids": unsolved_ids,
        "no_output_ids": no_output_ids,
    })

print(f"{'Lens':<15} {'Total':>5} {'Ran':>5} {'Solved':>7} {'Dual':>6} {'NoOut':>6} {'Solve%':>8} {'Dual%':>8} {'NoOut%':>8}")
print("-" * 90)

for r in lens_rows:
    print(
        f"{r['lens']:<15} {r['total']:>5} {r['ran']:>5} "
        f"{r['solved']:>7} {r['dual']:>6} {r['no_output']:>6} "
        f"{r['solve_rate']:>7.1f}% {r['dual_rate']:>7.1f}% {r['no_output_rate']:>7.1f}%"
    )

print("\nLens failure details:")
for r in lens_rows:
    if r["unsolved_ids"] or r["no_output_ids"]:
        print(f"\n{r['lens']}:")
        print(f"  unsolved : {r['unsolved_ids']}")
        print(f"  no_output: {r['no_output_ids']}")

print("\n" + "=" * 100)
print("FINAL DECISION GUARD FOR MAIN_DEBUG_48_EVAL_PROXY")
print("=" * 100)

lens_map = {r["lens"]: r for r in lens_rows}

print("Critical lenses to protect:")
for name in ["ALL_48", "TIER1", "EASY_STATIC"]:
    r = lens_map[name]
    print(
        f"  {name:<12}: solved={r['solved']}/{r['ran']} "
        f"dual={r['dual']}/{r['ran']} no_output={r['no_output']}/{r['total']}"
    )

print("\nWatch lenses:")
for name in ["RARE", "MULTI_QUERY", "DYNAMIC", "TIER3"]:
    r = lens_map[name]
    print(
        f"  {name:<12}: solved={r['solved']}/{r['ran']} "
        f"dual={r['dual']}/{r['ran']} no_output={r['no_output']}/{r['total']}"
    )

print("\nDecision rule:")
print("  KEEP if ALL_48 improves AND TIER1 does not drop AND EASY_STATIC does not drop AND no_output does not increase.")
print("  HOLD/REJECT if gain comes mainly from RARE/MULTI_QUERY while TIER1 or EASY_STATIC drops.")
print("  HOLD/REJECT if no_output increases, even when solved count improves.")

print("\n" + "=" * 100)
print("FINAL SUMMARY: TIER vs SOLVE RATE")
print("=" * 100)

tier_solve = {
    1: {"total": 0, "solved": 0},
    2: {"total": 0, "solved": 0},
    3: {"total": 0, "solved": 0},
}

for row in summary_data:
    if is_no_output(row):
        continue
    if row["id"] in data_dict:
        tier, _ = get_puzzle_priority(row["id"], data_dict)
        tier_solve[tier]["total"] += 1
        if is_solved(row):
            tier_solve[tier]["solved"] += 1

for tier in [1, 2, 3]:
    total_t = tier_solve[tier]["total"]
    solved_t = tier_solve[tier]["solved"]
    if total_t > 0:
        rate = solved_t / total_t * 100
        print(f"Tier {tier}: {solved_t}/{total_t} solved ({rate:.1f}%)")
    else:
        print(f"Tier {tier}: no ran puzzles in debug set")

print("=" * 100)
print("DIAGNOSTIC COMPLETE.")
print("=" * 100)


CHECK 1: SEED DETERMINISM
0934a4d8: train_seed=1767, eval_seed=7602
135a2760: train_seed=6686, eval_seed=5286
136b0064: train_seed=8451, eval_seed=2965
13e47133: train_seed=5706, eval_seed=2015
142ca369: train_seed=3462, eval_seed=3356
16b78196: train_seed=9181, eval_seed=5092
16de56c4: train_seed=5130, eval_seed=10
1818057f: train_seed=8477, eval_seed=111
195c6913: train_seed=5572, eval_seed=6828
1ae2feb7: train_seed=9959, eval_seed=5076
20270e3b: train_seed=4787, eval_seed=1336
20a9e565: train_seed=9481, eval_seed=3275
21897d95: train_seed=5987, eval_seed=7078
221dfab4: train_seed=4625, eval_seed=7016
247ef758: train_seed=7723, eval_seed=670
269e22fb: train_seed=9501, eval_seed=9308
271d71e2: train_seed=1368, eval_seed=128
28a6681f: train_seed=607, eval_seed=9328
291dc1e1: train_seed=3815, eval_seed=5128
2b83f449: train_seed=1322, eval_seed=6105
2ba387bc: train_seed=5761, eval_seed=1681
2c181942: train_seed=9182, eval_seed=7443
2d0172a1: train_seed=1789, eval_seed=1016
31f7f899: trai

In [11]:
# ================================================================
# CHECK 9-11: ROOT-CAUSE BUCKETS, FIX PRIORITY, RUNTIME/BUDGET EDA
# ================================================================

print("\n" + "=" * 100)
print("CHECK 9: PROBLEM BUCKETS")
print("=" * 100)

def row_by_id(pid):
    return summary_by_id.get(pid)

def feat_by_id(pid):
    return feature_by_id.get(pid, {})

all_ids = [r["id"] for r in summary_data]

no_output_ids = [pid for pid in all_ids if is_no_output(row_by_id(pid))]
solved_not_dual_ids = [
    pid for pid in all_ids
    if is_solved(row_by_id(pid)) and not dual_yes(row_by_id(pid))
]
unsolved_ids = [pid for pid in all_ids if is_unsolved(row_by_id(pid))]

unsolved_dynamic_ids = [
    pid for pid in unsolved_ids
    if feat_by_id(pid).get("dynamic", False)
]
unsolved_high_ratio_ids = [
    pid for pid in unsolved_ids
    if feat_by_id(pid).get("high_ratio", False)
]
unsolved_extreme_ratio_ids = [
    pid for pid in unsolved_ids
    if feat_by_id(pid).get("extreme_ratio", False)
]
unsolved_tier3_ids = [
    pid for pid in unsolved_ids
    if feat_by_id(pid).get("tier") == 3
]
unsolved_easy_static_ids = [
    pid for pid in unsolved_ids
    if "easy_static" in feat_by_id(pid).get("tags", [])
]

problem_buckets = {
    "NO_OUTPUT": no_output_ids,
    "SOLVED_NOT_DUAL": solved_not_dual_ids,
    "UNSOLVED_DYNAMIC": unsolved_dynamic_ids,
    "UNSOLVED_HIGH_RATIO": unsolved_high_ratio_ids,
    "UNSOLVED_EXTREME_RATIO": unsolved_extreme_ratio_ids,
    "UNSOLVED_TIER3": unsolved_tier3_ids,
    "UNSOLVED_EASY_STATIC": unsolved_easy_static_ids,
}

for name, ids in problem_buckets.items():
    print(f"{name:<24} {len(ids):>3}: {ids}")

print("\nInterpretation:")
print("  NO_OUTPUT          = infrastructure/timeout/decode-save issue.")
print("  SOLVED_NOT_DUAL    = selection issue; correct candidate exists but final dual missed it.")
print("  UNSOLVED_*         = generation/capability issue; correct candidate absent or not detected.")


print("\n" + "=" * 100)
print("CHECK 10: FIX PRIORITY SUGGESTION")
print("=" * 100)

priority_notes = []

if len(no_output_ids) > 0:
    priority_notes.append((
        "P1",
        "NO OUTPUT recovery",
        f"{len(no_output_ids)} puzzles have no output. First classify timeout vs finished-empty vs missing-worker-log."
    ))

if len(solved_not_dual_ids) > 0:
    priority_notes.append((
        "P2",
        "Selection check",
        f"{len(solved_not_dual_ids)} puzzles are solved in candidate pool but not selected by dual output."
    ))

dynamic_pressure = len(set(unsolved_dynamic_ids + unsolved_high_ratio_ids + unsolved_tier3_ids))
if dynamic_pressure > 0:
    priority_notes.append((
        "P3",
        "Dynamic/high-ratio/tier3 generation",
        f"{dynamic_pressure} hard structural puzzles remain unsolved. Need more candidate generation or specialized solver."
    ))

if len(unsolved_easy_static_ids) > 0:
    priority_notes.append((
        "P4",
        "Easy-static regression guard",
        f"{len(unsolved_easy_static_ids)} easy_static puzzles are unsolved. Do not sacrifice this group for rare/dynamic gains."
    ))

if not priority_notes:
    priority_notes.append((
        "OK",
        "No obvious priority bucket",
        "Main diagnostic buckets are clean. Next inspect per-puzzle candidates and runtime."
    ))

for level, title, msg in priority_notes:
    print(f"{level:<3} {title:<36} {msg}")

print("\nRecommended order:")
print("  1. Fix/classify NO_OUTPUT first.")
print("  2. If SOLVED_NOT_DUAL exists, verify actual run_dual_selection output.")
print("  3. If dynamic/high_ratio/tier3 unsolved dominates, improve generation not selection.")
print("  4. Protect easy_static; it is the safest LB-supporting group.")


print("\n" + "=" * 100)
print("CHECK 11: RUNTIME / BUDGET WASTE ANALYSIS")
print("=" * 100)
print("Paste full worker log into worker_log_text below to enable this check.")

worker_log_text = """
PASTE_FULL_WORKER_LOG_HERE
"""

finished_times = {}
timeout_times = {}

for match in re.finditer(r"\[Rank \d+\].*finished (\w+) in ([\d.]+)s", worker_log_text):
    finished_times[match.group(1)] = float(match.group(2))

for match in re.finditer(r"\[Rank \d+\].*timeout after ([\d.]+)s for puzzle (\w+)", worker_log_text):
    timeout_times[match.group(2)] = float(match.group(1))

if not finished_times and not timeout_times:
    print("No worker timing data found. Paste full starter/worker log into worker_log_text.")
else:
    print(f"Finished timing rows: {len(finished_times)}")
    print(f"Timeout rows:         {len(timeout_times)}")

    print("\nNO_OUTPUT root-cause classification:")
    for pid in no_output_ids:
        if pid in timeout_times:
            print(f"  {pid}: TIMEOUT confirmed at {timeout_times[pid]:.1f}s")
        elif pid in finished_times:
            print(
                f"  {pid}: FINISHED at {finished_times[pid]:.1f}s but NO OUTPUT "
                f"=> likely empty decoded_result / parse failure / save issue"
            )
        else:
            print(
                f"  {pid}: NO worker log entry "
                f"=> possible worker crash, queue delivery issue, or missing log"
            )

    timed_rows = []
    for pid, sec in finished_times.items():
        if pid not in summary_by_id:
            continue
        row = summary_by_id[pid]
        feat = feat_by_id(pid)
        timed_rows.append({
            "id": pid,
            "time": sec,
            "status": row["status"],
            "dual": row["solved_by_dual"],
            "tier": feat.get("tier", "N/A"),
            "dynamic": feat.get("dynamic", False),
            "high_ratio": feat.get("high_ratio", False),
            "easy_static": "easy_static" in feat.get("tags", []),
        })

    slow_rows = sorted(timed_rows, key=lambda x: x["time"], reverse=True)[:15]

    print("\nSlowest finished puzzles:")
    print(f"{'Puzzle':<12} {'Time':>8} {'Status':<10} {'Dual':<5} {'Tier':<5} {'Dyn':<5} {'HighR':<6} {'EasyStatic':<10}")
    print("-" * 80)
    for r in slow_rows:
        print(
            f"{r['id']:<12} {r['time']:>8.1f} {r['status']:<10} {r['dual']:<5} "
            f"{str(r['tier']):<5} {str(r['dynamic']):<5} {str(r['high_ratio']):<6} {str(r['easy_static']):<10}"
        )

    slow_unsolved = [r["id"] for r in timed_rows if r["time"] >= 1000 and is_unsolved(summary_by_id[r["id"]])]
    slow_solved = [r["id"] for r in timed_rows if r["time"] >= 1000 and is_solved(summary_by_id[r["id"]])]

    print("\nBudget signal:")
    print(f"  Slow solved   >=1000s: {len(slow_solved)} -> {slow_solved}")
    print(f"  Slow unsolved >=1000s: {len(slow_unsolved)} -> {slow_unsolved}")
    print("  If slow_unsolved dominates, extra time alone may be wasteful; need better candidate generation.")
    print("  If timeout/no_output dominates, recovery/partial-save logic should be tested first.")

print("=" * 100)
print("CHECK 9-11 COMPLETE.")
print("=" * 100)


CHECK 9: PROBLEM BUCKETS
NO_OUTPUT                  0: []
SOLVED_NOT_DUAL            8: ['221dfab4', '2b83f449', '3dc255db', '58490d8a', '6e453dd6', '8b7bacbf', 'b5ca7ac4', 'dfadab01']
UNSOLVED_DYNAMIC          25: ['0934a4d8', '136b0064', '20270e3b', '20a9e565', '21897d95', '269e22fb', '291dc1e1', '4c7dc4dd', '4e34c42c', '5545f144', '5dbc8537', '67e490f4', '6ffbe589', '7b3084d4', '7b5033c1', '8698868d', '89565ca0', '898e7135', 'a32d8b75', 'b0039139', 'e87109e9', 'edb79dae', 'f560132c', 'f931b4a8', 'fc7cae8d']
UNSOLVED_HIGH_RATIO       12: ['0934a4d8', '20a9e565', '4c7dc4dd', '4e34c42c', '67e490f4', '7b3084d4', '7b5033c1', '89565ca0', 'b0039139', 'f560132c', 'f931b4a8', 'fc7cae8d']
UNSOLVED_EXTREME_RATIO     4: ['0934a4d8', '20a9e565', '4c7dc4dd', '89565ca0']
UNSOLVED_TIER3             4: ['0934a4d8', '20a9e565', '4c7dc4dd', '89565ca0']
UNSOLVED_EASY_STATIC       7: ['271d71e2', '332f06d7', '409aa875', 'cb2d8a2c', 'da515329', 'db695cfb', 'dd6b8c4b']

Interpretation:
  NO_OUTPUT       

In [12]:
# ================================================================
# DYNAMIC PROOF: Rank 2 contribution & Rank 3-5 rescue analysis
# Runs on ALL decoded subkeys (no hardcoded targets)
# ================================================================
from arc_decoder import score_full_probmul_3, score_kgmon, hashable
import numpy as np
from collections import defaultdict

print("=" * 100)
print("PROOF: RANK 2 CONTRIBUTION & RANK 3-5 RESCUE ANALYSIS (FULL DATASET)")
print("=" * 100)

# ---- 1. Compute current dual score and probmul score dynamically ----
print("\n[1] COMPUTING CURRENT SCORES (from actual decoded results)")

# Weighted scoring function
def compute_score_for_policy(policy_fn):
    """policy_fn(bk, guesses_dict) -> list of selected candidates"""
    total_score = 0.0
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1

    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        picks = policy_fn(bk, guesses_dict)
        if any(np.array_equal(p, correct) for p in picks):
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
    return total_score

# Current dual selection
current_dual = decoder.run_dual_selection()
def dual_policy(bk, guesses_dict):
    return current_dual.get(bk, [])

# Probmul top1 only (Attempt1)
def probmul_top1_policy(bk, guesses_dict):
    g = {k: v for k, v in guesses_dict.items()}
    ranked_a = score_full_probmul_3(g)
    return [ranked_a[0]] if ranked_a else []

# Probmul top2 (Attempt1 + Attempt2 = Rank2)
def probmul_top2_policy(bk, guesses_dict):
    g = {k: v for k, v in guesses_dict.items()}
    ranked_a = score_full_probmul_3(g)
    return ranked_a[:2] if len(ranked_a) >= 2 else ranked_a

# Kgmon top2
def kgmon_top2_policy(bk, guesses_dict):
    g = {k: v for k, v in guesses_dict.items()}
    ranked_b = score_kgmon(g)
    return ranked_b[:2] if len(ranked_b) >= 2 else ranked_b

score_dual = compute_score_for_policy(dual_policy)
score_probmul_top1 = compute_score_for_policy(probmul_top1_policy)
score_probmul_top2 = compute_score_for_policy(probmul_top2_policy)
score_kgmon_top2 = compute_score_for_policy(kgmon_top2_policy)

print(f"  Probmul Top1 (Attempt1 only)   : {score_probmul_top1:.2f}")
print(f"  Probmul Top2 (Attempt1+Rank2)  : {score_probmul_top2:.2f}")
print(f"  Kgmon Top2                     : {score_kgmon_top2:.2f}")
print(f"  Current Dual (Attempt1+Attempt2): {score_dual:.2f}")

rank2_gain = score_probmul_top2 - score_probmul_top1
print(f"\n  Rank 2 gain (Probmul Top2 - Top1): {rank2_gain:+.2f}")
if rank2_gain == 0:
    print("  ✅ Rank 2 adds ZERO new solves (identical to Top1 only).")
else:
    print(f"  ⚠️  Rank 2 adds {rank2_gain:.2f} points.")

dual_gain_over_probmul_top1 = score_dual - score_probmul_top1
print(f"  Dual gain over Probmul Top1   : {dual_gain_over_probmul_top1:+.2f}")
if dual_gain_over_probmul_top1 == 0:
    print("  ✅ Dual selection adds NO new solves beyond Probmul Top1.")
else:
    print(f"  ⚠️  Dual adds {dual_gain_over_probmul_top1:.2f} points.")

# ---- 2. Examine all puzzles: find GT present but not selected by dual ----
print("\n[2] ANALYZING ALL PUZZLES (GT presence & rank distribution)")

# Collect stats
all_stats = []
rescue_candidates = []

for bk, guesses_dict in decoder.decoded_results.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue

    g = {k: v for k, v in guesses_dict.items()}
    ranked_a = score_full_probmul_3(g)
    ranked_b = score_kgmon(g)

    rank_a = None
    rank_b = None
    for i, cand in enumerate(ranked_a):
        if np.array_equal(cand, correct):
            rank_a = i + 1
            break
    for i, cand in enumerate(ranked_b):
        if np.array_equal(cand, correct):
            rank_b = i + 1
            break

    gt_present = (rank_a is not None) or (rank_b is not None)
    if not gt_present:
        continue

    # Check if dual selects it
    dual_picks = current_dual.get(bk, [])
    dual_correct = any(np.array_equal(p, correct) for p in dual_picks)

    best_rank = min(r for r in [rank_a, rank_b] if r is not None)

    all_stats.append({
        "bk": bk,
        "rank_a": rank_a,
        "rank_b": rank_b,
        "best_rank": best_rank,
        "dual_correct": dual_correct,
    })

    if not dual_correct and best_rank in (3,4,5):
        rescue_candidates.append((bk, best_rank))

print(f"\nTotal subkeys with GT present: {len(all_stats)}")
if len(all_stats) == 0:
    print("⚠️  No GT found in any candidate pool – check decoding.")
else:
    # Group by best rank
    rank_counts = defaultdict(int)
    dual_fail_counts = defaultdict(int)
    for s in all_stats:
        rank_counts[s["best_rank"]] += 1
        if not s["dual_correct"]:
            dual_fail_counts[s["best_rank"]] += 1

    print("\nGT Rank Distribution (best rank among Probmul/Kgmon):")
    for r in sorted(rank_counts.keys()):
        total = rank_counts[r]
        fail = dual_fail_counts.get(r, 0)
        print(f"  Rank {r}: {total} puzzles, {fail} not selected by dual")

    # Show rescue candidates (rank 3-5 not selected)
    print(f"\n[3] PUZZLES THAT WILL BE RESCUED BY RANK 3-5 PICK")
    if rescue_candidates:
        for bk, r in rescue_candidates:
            print(f"  ✅ {bk}  (best rank = {r})")
        print(f"\n   Total {len(rescue_candidates)} subkeys with GT in ranks 3-5 but missed by dual.")
        print("   These can be solved by picking best candidate from ranks 3-5 for Attempt 2.")
    else:
        print("   No rescue candidates found – all GT in ranks 3-5 are already selected by dual.")

    # Compute theoretical max gain if we could always pick GT when it exists
    # (Oracle: always pick GT if present)
    oracle_score = 0.0
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        g = {k: v for k, v in guesses_dict.items()}
        ranked_a = score_full_probmul_3(g)
        rank_a = None
        for i, cand in enumerate(ranked_a):
            if np.array_equal(cand, correct):
                rank_a = i + 1
                break
        if rank_a is not None:
            # Oracle can always pick it
            puzzle_id = bk.rsplit("_", 1)[0]
            oracle_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

    print(f"\n[4] THEORETICAL UPPER BOUND (Oracle selection)")
    print(f"  Current Dual Score: {score_dual:.2f}")
    print(f"  Oracle Score       : {oracle_score:.2f}")
    print(f"  Potential Gain     : {oracle_score - score_dual:+.2f}")
    print(f"  (Oracle assumes we can always pick GT when it exists in pool)")

    print("\n[5] FINAL VERDICT")
    if rank2_gain == 0:
        print("  ✅ Rank 2 adds zero solves – skipping it is 100% safe.")
    else:
        print(f"  ⚠️  Rank 2 adds {rank2_gain:.2f} solves – removing it may lose those.")
    print(f"  Selecting best from ranks 3-5 for Attempt 2 could rescue up to {len(rescue_candidates)} subkeys.")
    print("  This is a data-driven fix that directly targets the observed bottleneck.")

print("\n" + "=" * 100)
print("✅ ANALYSIS COMPLETE.")
print("=" * 100)

PROOF: RANK 2 CONTRIBUTION & RANK 3-5 RESCUE ANALYSIS (FULL DATASET)

[1] COMPUTING CURRENT SCORES (from actual decoded results)
  Probmul Top1 (Attempt1 only)   : 26.00
  Probmul Top2 (Attempt1+Rank2)  : 33.50
  Kgmon Top2                     : 34.83
  Current Dual (Attempt1+Attempt2): 33.50

  Rank 2 gain (Probmul Top2 - Top1): +7.50
  ⚠️  Rank 2 adds 7.50 points.
  Dual gain over Probmul Top1   : +7.50
  ⚠️  Dual adds 7.50 points.

[2] ANALYZING ALL PUZZLES (GT presence & rank distribution)

Total subkeys with GT present: 55

GT Rank Distribution (best rank among Probmul/Kgmon):
  Rank 1: 37 puzzles, 0 not selected by dual
  Rank 2: 10 puzzles, 2 not selected by dual
  Rank 3: 4 puzzles, 4 not selected by dual
  Rank 5: 3 puzzles, 3 not selected by dual
  Rank 9: 1 puzzles, 1 not selected by dual

[3] PUZZLES THAT WILL BE RESCUED BY RANK 3-5 PICK
  ✅ 58490d8a_0  (best rank = 5)
  ✅ 221dfab4_1  (best rank = 5)
  ✅ b5ca7ac4_0  (best rank = 3)
  ✅ 2b83f449_0  (best rank = 3)
  ✅ 3dc255

In [13]:
# ================================================================
# SAFE DIAGNOSTIC: Test Rank 3, Rank 4, Rank 5 as Attempt 2
# (Does NOT change run_dual_selection - purely experimental)
# ================================================================
from arc_decoder import score_full_probmul_3, hashable
import numpy as np

def evaluate_attempt2_variant(rank_index):
    """
    rank_index: 2 for Rank 3, 3 for Rank 4, 4 for Rank 5 (0-based)
    """
    total_score = 0.0
    num_tasks_per_puzzle = {}
    
    # First pass: count tasks per puzzle for weighted scoring
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] = num_tasks_per_puzzle.get(puzzle_id, 0) + 1
    
    correct_count = 0
    total_puzzles = len(decoder.decoded_results)
    
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        
        # Use Probmul ranking (safest primary signal)
        ranked = score_full_probmul_3(guesses_dict)
        
        # Attempt 1: always top pick
        attempt1 = ranked[0] if ranked else None
        
        # Attempt 2: pick the specified rank
        attempt2 = ranked[rank_index] if len(ranked) > rank_index else None
        
        # Check if either attempt matches correct
        solved = False
        if attempt1 is not None and np.array_equal(attempt1, correct):
            solved = True
        elif attempt2 is not None and np.array_equal(attempt2, correct):
            solved = True
        
        if solved:
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            correct_count += 1
    
    return total_score, correct_count, total_puzzles

print("=" * 100)
print("EXPERIMENT: Which Rank (3, 4, or 5) gives the best score?")
print("=" * 100)

# Current Dual baseline (for comparison)
current_dual_results = decoder.run_dual_selection()
current_score = 0.0
num_tasks_per_puzzle = {}
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    num_tasks_per_puzzle[puzzle_id] = num_tasks_per_puzzle.get(puzzle_id, 0) + 1

for bk, picks in current_dual_results.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue
    if any(np.array_equal(p, correct) for p in picks):
        puzzle_id = bk.rsplit("_", 1)[0]
        current_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

print(f"\nBaseline (Current Dual - Top1 + Top2 from Pool): {current_score:.2f}")

# Test Rank 3, 4, 5
for rank_name, rank_idx in [("Rank 3", 2), ("Rank 4", 3), ("Rank 5", 4)]:
    score, correct, total = evaluate_attempt2_variant(rank_idx)
    print(f"\n{rank_name} as Attempt 2:")
    print(f"  Score     : {score:.2f}")
    print(f"  Solved    : {correct}/{total} puzzles")
    print(f"  Change    : {'+' if score > current_score else ''}{score - current_score:.2f}")

print("\n" + "=" * 100)
print("RECOMMENDATION:")
print("  If Rank 4 wins, we simply set Attempt2 = ranked_a[3] in run_dual_selection().")
print("  If Rank 3 wins, we set Attempt2 = ranked_a[2].")
print("  If Rank 5 wins, we set Attempt2 = ranked_a[4].")
print("  This keeps the change minimal and directly targets the evidence.")
print("=" * 100)

EXPERIMENT: Which Rank (3, 4, or 5) gives the best score?

Baseline (Current Dual - Top1 + Top2 from Pool): 33.50

Rank 3 as Attempt 2:
  Score     : 30.67
  Solved    : 42/172 puzzles
  Change    : -2.83

Rank 4 as Attempt 2:
  Score     : 26.00
  Solved    : 36/172 puzzles
  Change    : -7.50

Rank 5 as Attempt 2:
  Score     : 28.50
  Solved    : 39/172 puzzles
  Change    : -5.00

RECOMMENDATION:
  If Rank 4 wins, we simply set Attempt2 = ranked_a[3] in run_dual_selection().
  If Rank 3 wins, we set Attempt2 = ranked_a[2].
  If Rank 5 wins, we set Attempt2 = ranked_a[4].
  This keeps the change minimal and directly targets the evidence.


In [14]:
# ================================================================
# SAFE DIAGNOSTIC: Test Rank 3, Rank 4, Rank 5 as Attempt 2
# ================================================================
from arc_decoder import score_full_probmul_3, hashable
import numpy as np

def evaluate_attempt2_variant(rank_index):
    total_score = 0.0
    num_tasks_per_puzzle = {}
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] = num_tasks_per_puzzle.get(puzzle_id, 0) + 1
    
    correct_count = 0
    total_puzzles = len(decoder.decoded_results)
    
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        ranked = score_full_probmul_3(guesses_dict)
        attempt1 = ranked[0] if ranked else None
        attempt2 = ranked[rank_index] if len(ranked) > rank_index else None
        
        solved = False
        if attempt1 is not None and np.array_equal(attempt1, correct):
            solved = True
        elif attempt2 is not None and np.array_equal(attempt2, correct):
            solved = True
        
        if solved:
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            correct_count += 1
    
    return total_score, correct_count, total_puzzles

print("=" * 100)
print("EXPERIMENT: Which Rank (3, 4, or 5) gives the best score?")
print("=" * 100)

current_dual_results = decoder.run_dual_selection()
current_score = 0.0
num_tasks_per_puzzle = {}
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    num_tasks_per_puzzle[puzzle_id] = num_tasks_per_puzzle.get(puzzle_id, 0) + 1

for bk, picks in current_dual_results.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue
    if any(np.array_equal(p, correct) for p in picks):
        puzzle_id = bk.rsplit("_", 1)[0]
        current_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

print(f"\nBaseline (Current Dual): {current_score:.2f}")

for rank_name, rank_idx in [("Rank 3", 2), ("Rank 4", 3), ("Rank 5", 4)]:
    score, correct, total = evaluate_attempt2_variant(rank_idx)
    print(f"\n{rank_name} as Attempt 2:")
    print(f"  Score     : {score:.2f}")
    print(f"  Solved    : {correct}/{total} puzzles")
    print(f"  Change    : {'+' if score > current_score else ''}{score - current_score:.2f}")

print("\n" + "=" * 100)

EXPERIMENT: Which Rank (3, 4, or 5) gives the best score?

Baseline (Current Dual): 33.50

Rank 3 as Attempt 2:
  Score     : 30.67
  Solved    : 42/172 puzzles
  Change    : -2.83

Rank 4 as Attempt 2:
  Score     : 26.00
  Solved    : 36/172 puzzles
  Change    : -7.50

Rank 5 as Attempt 2:
  Score     : 28.50
  Solved    : 39/172 puzzles
  Change    : -5.00



In [15]:
import json
import numpy as np

GT_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"

with open(GT_PATH) as f:
    GT = json.load(f)


def same_grid(a, b):
    return np.array_equal(np.asarray(a), np.asarray(b))


def find_rank(ranked_candidates, gt):
    for rank, cand in enumerate(ranked_candidates, start=1):
        if same_grid(cand, gt):
            return rank
    return None

print("=" * 120)

# Collect all puzzle IDs from decoded_results
all_puzzle_ids = set()
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    all_puzzle_ids.add(puzzle_id)

print(f"Total puzzles found in decoded_results: {len(all_puzzle_ids)}")
print("-" * 120)

for task_id in sorted(all_puzzle_ids):
    # Find all decoded subkeys that belong to this puzzle
    matching_bks = [k for k in decoder.decoded_results.keys() if k.startswith(task_id)]

    if not matching_bks:
        print(f"\n⚠️  No decoded results found for {task_id}")
        continue

    # Sort to process _0, _1, _2... in order
    matching_bks.sort()

    print(f"\n######## {task_id} ########")

    for bk in matching_bks:
        # Extract sub-index (e.g., "4c7dc4dd_1" -> sub_idx = 1)
        sub_idx = int(bk.rsplit("_", 1)[1])

        # Get the correct ground truth for this specific sub-index
        if task_id not in GT:
            print(f"\n⚠️  No GT found for {task_id}")
            continue

        if sub_idx >= len(GT[task_id]):
            print(f"\n⚠️  GT missing for {bk} (sub_idx={sub_idx})")
            continue

        gt = GT[task_id][sub_idx]

        # Get the decoded guesses for this specific subkey
        guesses = decoder.decoded_results[bk]
        guesses = {k: v for k, v in guesses.items()}

        ranked_prob = score_full_probmul_3(guesses)
        ranked_kg = score_kgmon(guesses)

        r1 = find_rank(ranked_prob, gt)
        r2 = find_rank(ranked_kg, gt)

        print(f"\nSubkey: {bk}")
        print(f"Probmul rank : {r1}")
        print(f"Kgmon   rank : {r2}")

        if r1 is not None:
            print("✓ Probmul found GT")
        else:
            print("✗ Probmul missed GT")

        if r2 is not None:
            print("✓ Kgmon found GT")
        else:
            print("✗ Kgmon missed GT")

print("\nDone.")

Total puzzles found in decoded_results: 120
------------------------------------------------------------------------------------------------------------------------

######## 0934a4d8 ########

Subkey: 0934a4d8_0
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

######## 135a2760 ########

Subkey: 135a2760_0
Probmul rank : 1
Kgmon   rank : 1
✓ Probmul found GT
✓ Kgmon found GT

######## 136b0064 ########

Subkey: 136b0064_0
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

######## 13e47133 ########

Subkey: 13e47133_0
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

Subkey: 13e47133_1
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

######## 142ca369 ########

Subkey: 142ca369_0
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

Subkey: 142ca369_1
Probmul rank : None
Kgmon   rank : None
✗ Probmul missed GT
✗ Kgmon missed GT

######## 16b7819

In [16]:
import json
import numpy as np

GT_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"

with open(GT_PATH) as f:
    GT = json.load(f)


def same_grid(a, b):
    return np.array_equal(np.asarray(a), np.asarray(b))


def find_rank(ranked_candidates, gt):
    for rank, cand in enumerate(ranked_candidates, start=1):
        if same_grid(cand, gt):
            return rank
    return None


def grid_similarity(a, b):
    """Similarity percentage (0-100)"""
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        return 0.0
    return np.mean(a == b) * 100


def grid_diff_count(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        return -1
    return np.sum(a != b)


def diff_coords(a, b, max_display=15):
    """Coordinates of differing cells (shows up to max_display)"""
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        return []
    coords = np.argwhere(a != b)
    if len(coords) > max_display:
        return coords[:max_display].tolist() + ['...']
    return coords.tolist()


def get_best_candidate(guesses, gt):
    """Finds the candidate most similar to GT"""
    best_sim = -1.0
    best_cand = None
    best_idx = -1
    for idx, (key, val) in enumerate(guesses.items()):
        cand = val["solution"]
        sim = grid_similarity(cand, gt)
        if sim > best_sim:
            best_sim = sim
            best_cand = cand
            best_idx = idx
    return best_cand, best_sim, best_idx


# Collect all puzzle IDs from decoded_results
all_puzzle_ids = set()
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    all_puzzle_ids.add(puzzle_id)

print("=" * 120)
print("DETAILED CANDIDATE ANALYSIS (Actionable Insights) - FULL DATASET")
print("=" * 120)

for task_id in sorted(all_puzzle_ids):
    matching_bks = [k for k in decoder.decoded_results.keys() if k.startswith(task_id)]

    if not matching_bks:
        print(f"\n⚠️  No decoded results found for {task_id}")
        continue

    matching_bks.sort()

    print(f"\n######## {task_id} ########")

    for bk in matching_bks:
        sub_idx = int(bk.rsplit("_", 1)[1])

        if task_id not in GT or sub_idx >= len(GT[task_id]):
            print(f"\n⚠️  GT missing for {bk}")
            continue

        gt = GT[task_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        guesses_dict = {k: v for k, v in guesses.items()}

        ranked_prob = score_full_probmul_3(guesses_dict)
        ranked_kg = score_kgmon(guesses_dict)

        gt_present = find_rank(ranked_prob, gt) is not None

        best_cand, best_sim, best_idx = get_best_candidate(guesses_dict, gt)

        print(f"\n📌 Subkey: {bk}")
        print(f"   Total candidates generated : {len(guesses_dict)}")

        if best_cand is not None:
            diff_count = grid_diff_count(best_cand, gt)
            print(f"   Best candidate similarity    : {best_sim:.2f}%")
            print(f"   Pixel differences (best)     : {diff_count}")
            if diff_count > 0 and diff_count <= 20:
                coords = diff_coords(best_cand, gt)
                print(f"   Differing cells (up to 15)   : {coords}")
            elif diff_count > 20:
                print(f"   Differing cells              : Too many ({diff_count}) to list")
        else:
            print("   Best candidate similarity    : N/A (no candidates)")

        r1 = find_rank(ranked_prob, gt)
        r2 = find_rank(ranked_kg, gt)

        print(f"   Probmul rank                 : {r1 if r1 is not None else 'Not found'}")
        print(f"   Kgmon rank                   : {r2 if r2 is not None else 'Not found'}")

        if gt_present:
            print("   ✅ Ground truth is PRESENT in candidate pool.")
        else:
            print("   ❌ Ground truth is MISSING from candidate pool.")
            if best_cand is not None and best_sim < 100:
                print(f"   🔍 Closest candidate is {best_sim:.2f}% similar to GT.")
                if best_sim < 50:
                    print("   ⚠️  Generation is severely off — model is producing unrelated outputs.")
                elif best_sim < 90:
                    print("   ⚠️  Generation is partially correct — needs fine-tuning or better augmentation.")
                else:
                    print("   ✅ Generation is very close (≥90%) but exact match missing — minor edits or selection tweak needed.")

print("\n" + "=" * 120)
print("ANALYSIS COMPLETE.")
print("=" * 120)

DETAILED CANDIDATE ANALYSIS (Actionable Insights) - FULL DATASET

######## 0934a4d8 ########

📌 Subkey: 0934a4d8_0
   Total candidates generated : 17
   Best candidate similarity    : 70.37%
   Pixel differences (best)     : 8
   Differing cells (up to 15)   : [[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1], [3, 0], [3, 1]]
   Probmul rank                 : Not found
   Kgmon rank                   : Not found
   ❌ Ground truth is MISSING from candidate pool.
   🔍 Closest candidate is 70.37% similar to GT.
   ⚠️  Generation is partially correct — needs fine-tuning or better augmentation.

######## 135a2760 ########

📌 Subkey: 135a2760_0
   Total candidates generated : 20
   Best candidate similarity    : 100.00%
   Pixel differences (best)     : 0
   Probmul rank                 : 1
   Kgmon rank                   : 1
   ✅ Ground truth is PRESENT in candidate pool.

######## 136b0064 ########

📌 Subkey: 136b0064_0
   Total candidates generated : 1
   Best candidate similarity    : 66.1

In [17]:
import json
import numpy as np
from collections import defaultdict

# ================================================================
# SELECTOR BENCHMARK: Test all policies against Ground Truth
# ================================================================

def evaluate_policy(policy_name, select_fn):
    """
    select_fn(ranked_a, ranked_b, all_unique_candidates) -> candidate_grid
    Returns (total_score, correct_count, total_puzzles)
    """
    total_score = 0.0
    correct_count = 0
    
    # Weighted scoring (1 / num_tasks_per_puzzle)
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1
    
    total_puzzles = len(decoder.decoded_results)
    
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        
        g = {k: v for k, v in guesses_dict.items()}
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        # Build unique candidate pool (union of both rankers)
        pool = {}
        for idx, cand in enumerate(ranked_a, start=1):
            h = hashable(cand)
            if h not in pool:
                pool[h] = {'cand': cand, 'rank_a': idx, 'rank_b': None}
            else:
                pool[h]['rank_a'] = idx
        for idx, cand in enumerate(ranked_b, start=1):
            h = hashable(cand)
            if h not in pool:
                pool[h] = {'cand': cand, 'rank_a': None, 'rank_b': idx}
            else:
                pool[h]['rank_b'] = idx
        
        # Attempt 1: always Probmul Top1 (unchanged)
        attempt1 = ranked_a[0] if ranked_a else None
        
        # Attempt 2: apply the selection policy
        if not pool:
            attempt2 = attempt1
        else:
            # Convert to list for selection function
            candidates = list(pool.values())
            attempt2 = select_fn(ranked_a, ranked_b, candidates)
        
        # Check if either attempt matches GT
        solved = False
        if attempt1 is not None and np.array_equal(attempt1, correct):
            solved = True
        elif attempt2 is not None and np.array_equal(attempt2, correct):
            solved = True
        
        if solved:
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            correct_count += 1
    
    return total_score, correct_count, total_puzzles


# --- Define Selection Policies ---

def policy_rank3(ranked_a, ranked_b, candidates):
    """Attempt2 = Probmul Rank 3"""
    return ranked_a[2] if len(ranked_a) > 2 else ranked_a[0]

def policy_rank4(ranked_a, ranked_b, candidates):
    """Attempt2 = Probmul Rank 4"""
    return ranked_a[3] if len(ranked_a) > 3 else ranked_a[0]

def policy_rank5(ranked_a, ranked_b, candidates):
    """Attempt2 = Probmul Rank 5"""
    return ranked_a[4] if len(ranked_a) > 4 else ranked_a[0]

def policy_rank_sum(ranked_a, ranked_b, candidates):
    """Minimize rank_a + rank_b (lower is better)"""
    best = None
    best_score = float('inf')
    for c in candidates:
        ra = c['rank_a'] if c['rank_a'] is not None else 1000
        rb = c['rank_b'] if c['rank_b'] is not None else 1000
        score = ra + rb
        if score < best_score:
            best_score = score
            best = c['cand']
    return best

def policy_minimax(ranked_a, ranked_b, candidates):
    """Minimize max(rank_a, rank_b) - most stable consensus"""
    best = None
    best_score = float('inf')
    for c in candidates:
        ra = c['rank_a'] if c['rank_a'] is not None else 1000
        rb = c['rank_b'] if c['rank_b'] is not None else 1000
        score = max(ra, rb)
        if score < best_score:
            best_score = score
            best = c['cand']
        elif score == best_score:
            # Tie-break: lower sum of ranks
            if (ra + rb) < (best_score + best_score):
                best = c['cand']
    return best

def policy_borda(ranked_a, ranked_b, candidates):
    """Borda count: sum of ranks (same as rank_sum, but explicit)"""
    return policy_rank_sum(ranked_a, ranked_b, candidates)

def policy_lex_prob(ranked_a, ranked_b, candidates):
    """Lexicographic: prioritize Probmul rank, then Kgmon rank"""
    best = None
    best_score = (float('inf'), float('inf'))
    for c in candidates:
        ra = c['rank_a'] if c['rank_a'] is not None else 1000
        rb = c['rank_b'] if c['rank_b'] is not None else 1000
        score = (ra, rb)
        if score < best_score:
            best_score = score
            best = c['cand']
    return best


# --- Run Benchmark ---

policies = [
    ("Current Dual (Rank1+Rank2)", None),  # We'll use decoder.run_dual_selection() for baseline
    ("Rank 3", policy_rank3),
    ("Rank 4", policy_rank4),
    ("Rank 5", policy_rank5),
    ("Rank Sum (Borda)", policy_rank_sum),
    ("Minimax (Max Rank)", policy_minimax),
    ("Lexicographic (Probmul first)", policy_lex_prob),
]

print("=" * 120)
print("SELECTOR BENCHMARK: Which Policy Actually Wins?")
print("=" * 120)

# Baseline: Current Dual Selection
current_dual = decoder.run_dual_selection()
base_score = 0.0
num_tasks_per_puzzle = defaultdict(int)
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    num_tasks_per_puzzle[puzzle_id] += 1

for bk, picks in current_dual.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue
    if any(np.array_equal(p, correct) for p in picks):
        puzzle_id = bk.rsplit("_", 1)[0]
        base_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

print(f"\n📊 Baseline (Current Dual): {base_score:.2f}\n")

results = []
for name, policy_fn in policies:
    if policy_fn is None:
        continue
    score, correct, total = evaluate_policy(name, policy_fn)
    change = score - base_score
    results.append((name, score, correct, total, change))
    print(f"✅ {name:<30} | Score: {score:5.2f} | Solved: {correct:2}/{total} | Change: {change:+.2f}")

# Find the best
best = max(results, key=lambda x: x[1])
print("\n" + "=" * 120)
print(f"🏆 WINNER: {best[0]} with score {best[1]:.2f}")
print("=" * 120)


SELECTOR BENCHMARK: Which Policy Actually Wins?

📊 Baseline (Current Dual): 33.50

✅ Rank 3                         | Score: 30.67 | Solved: 42/172 | Change: -2.83
✅ Rank 4                         | Score: 26.00 | Solved: 36/172 | Change: -7.50
✅ Rank 5                         | Score: 28.50 | Solved: 39/172 | Change: -5.00
✅ Rank Sum (Borda)               | Score: 26.00 | Solved: 36/172 | Change: -7.50
✅ Minimax (Max Rank)             | Score: 27.00 | Solved: 37/172 | Change: -6.50
✅ Lexicographic (Probmul first)  | Score: 26.00 | Solved: 36/172 | Change: -7.50

🏆 WINNER: Rank 3 with score 30.67


In [18]:
# ================================================================
# SETUP: Common functions and data for all experiments
# ================================================================
import json
import numpy as np
from collections import defaultdict, Counter
import pandas as pd

# Load ground truth if not already loaded
try:
    GT
except NameError:
    with open("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json") as f:
        GT = json.load(f)

def same_grid(a, b):
    return np.array_equal(np.asarray(a), np.asarray(b))

def hashable(grid):
    return tuple(map(tuple, np.asarray(grid)))

def find_rank(ranked_candidates, gt):
    for rank, cand in enumerate(ranked_candidates, start=1):
        if same_grid(cand, gt):
            return rank
    return None

def get_all_bks():
    """Return all base_keys with their subkeys"""
    bks = {}
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        bks.setdefault(puzzle_id, []).append(bk)
    return bks

ALL_BKS = get_all_bks()

In [19]:
# ================================================================
# EXPERIMENT 1: Candidate Feature Analysis (FIXED)
# ================================================================

print("=" * 120)
print("EXPERIMENT 1: GT Candidate Feature Analysis")
print("=" * 120)

features_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        rank_a = find_rank(ranked_a, gt)
        rank_b = find_rank(ranked_b, gt)
        
        gt_features = None
        for cand in g.values():
            if same_grid(cand["solution"], gt):
                gt_features = cand
                break
        
        current_dual = decoder.run_dual_selection().get(bk, [])
        attempt2 = current_dual[1] if len(current_dual) > 1 else None
        attempt2_features = None
        if attempt2 is not None:
            for cand in g.values():
                if same_grid(cand["solution"], attempt2):
                    attempt2_features = cand
                    break
        
        row = {
            "bk": bk,
            "rank_a": rank_a,
            "rank_b": rank_b,
            "gt_present": rank_a is not None or rank_b is not None,
        }
        
        if gt_features:
            row.update({
                "gt_beam": gt_features["beam_score"],
                "gt_aug_mean": np.mean(gt_features["score_aug"]),
                "gt_aug_min": min(gt_features["score_aug"]),
                "gt_aug_max": max(gt_features["score_aug"]),
            })
        else:
            row.update({
                "gt_beam": None,
                "gt_aug_mean": None,
                "gt_aug_min": None,
                "gt_aug_max": None,
            })
        
        if attempt2_features:
            row.update({
                "a2_beam": attempt2_features["beam_score"],
                "a2_aug_mean": np.mean(attempt2_features["score_aug"]),
            })
        else:
            row.update({
                "a2_beam": None,
                "a2_aug_mean": None,
            })
        
        features_data.append(row)

df_features = pd.DataFrame(features_data)

print("\n📊 Summary Statistics:")
print(f"Total subkeys analyzed: {len(df_features)}")
print(f"GT present in pool: {df_features['gt_present'].sum()}/{len(df_features)}")

# 🔧 FIX: Create an explicit copy to avoid SettingWithCopyWarning
df_gt = df_features[df_features['gt_present'] == True].copy()

if len(df_gt) > 0:
    print(f"\n📌 GT Present cases ({len(df_gt)}):")
    print(df_gt[['bk', 'rank_a', 'rank_b', 'gt_beam', 'gt_aug_mean']].to_string(index=False))
    
    print("\n📊 GT vs Attempt2 Feature Comparison:")
    print(f"  GT beam score mean:      {df_gt['gt_beam'].mean():.4f}")
    print(f"  Attempt2 beam score mean: {df_gt['a2_beam'].mean():.4f}")
    print(f"  GT aug score mean:       {df_gt['gt_aug_mean'].mean():.4f}")
    print(f"  Attempt2 aug score mean:  {df_gt['a2_aug_mean'].mean():.4f}")
    
    # 🔧 FIX: Now this won't trigger the warning
    df_gt['beam_diff'] = df_gt['gt_beam'] - df_gt['a2_beam']
    print(f"\n  Beam score diff (GT - A2): {df_gt['beam_diff'].mean():.4f}")
    print(f"  (Negative means GT has lower beam score = better)")
else:
    print("\n⚠️  No GT found in any candidate pool.")

print("=" * 120)

EXPERIMENT 1: GT Candidate Feature Analysis

📊 Summary Statistics:
Total subkeys analyzed: 172
GT present in pool: 55/172

📌 GT Present cases (55):
        bk  rank_a  rank_b  gt_beam  gt_aug_mean
1ae2feb7_2     1.0     1.0 0.010488     2.958130
d8e07eb2_1     1.0     1.0 1.145142     1.400799
78332cb0_1     1.0     1.0 0.433719     5.946105
e8686506_0     1.0     1.0 0.007671     4.622933
2d0172a1_0     1.0     1.0 1.038839     3.970051
2d0172a1_1     2.0     2.0 0.584828     3.971595
38007db0_0     1.0     1.0 0.000217     0.385819
38007db0_1     1.0     1.0 0.000266     0.000225
45a5af55_0     2.0     2.0 0.201438     3.325421
7ed72f31_0     1.0     1.0 0.490552     4.312501
7c66cb00_0     1.0     1.0 0.013628     0.881673
71e489b6_0     1.0     1.0 0.042282     5.805684
71e489b6_1     1.0     1.0 0.062457     2.600237
b6f77b65_1     1.0     1.0 0.250524     1.593739
b6f77b65_0     3.0     3.0 0.245835    12.782489
a6f40cea_0     1.0     2.0 1.217662     7.340729
53fb4810_0     1.0 

In [20]:
# ================================================================
# EXPERIMENT 2: Pool Coverage Analysis
# Total candidates, unique candidates, duplicate ratio, GT rank
# ================================================================

print("=" * 120)
print("EXPERIMENT 2: Pool Coverage Analysis")
print("=" * 120)

coverage_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        rank_a = find_rank(ranked_a, gt)
        rank_b = find_rank(ranked_b, gt)
        
        total_cands = len(g)
        unique_cands = len({hashable(c["solution"]) for c in g.values()})
        duplicate_ratio = 1 - (unique_cands / total_cands) if total_cands > 0 else 0
        
        # Which rankers found GT?
        found_by_prob = rank_a is not None
        found_by_kg = rank_b is not None
        
        coverage_data.append({
            "bk": bk,
            "total": total_cands,
            "unique": unique_cands,
            "dup_ratio": duplicate_ratio,
            "rank_a": rank_a,
            "rank_b": rank_b,
            "found_by_prob": found_by_prob,
            "found_by_kg": found_by_kg,
        })

df_coverage = pd.DataFrame(coverage_data)

print("\n📊 Overall Statistics:")
print(f"Total subkeys: {len(df_coverage)}")
print(f"Average candidates per subkey: {df_coverage['total'].mean():.1f}")
print(f"Average unique candidates: {df_coverage['unique'].mean():.1f}")
print(f"Average duplicate ratio: {df_coverage['dup_ratio'].mean():.3f}")

print("\n📊 GT Found By:")
print(f"  Probmul only:  {((df_coverage['found_by_prob'] == True) & (df_coverage['found_by_kg'] == False)).sum()}")
print(f"  Kgmon only:    {((df_coverage['found_by_prob'] == False) & (df_coverage['found_by_kg'] == True)).sum()}")
print(f"  Both:          {(df_coverage['found_by_prob'] == True) & (df_coverage['found_by_kg'] == True).sum()}")
print(f"  Neither:       {(df_coverage['found_by_prob'] == False) & (df_coverage['found_by_kg'] == False).sum()}")

print("\n📊 GT Rank Distribution (Probmul):")
print(df_coverage['rank_a'].value_counts().sort_index().to_string())

print("\n📊 GT Rank Distribution (Kgmon):")
print(df_coverage['rank_b'].value_counts().sort_index().to_string())

# Per-puzzle summary
print("\n📌 Per-puzzle GT Presence:")
for puzzle_id in sorted(ALL_BKS.keys()):
    df_p = df_coverage[df_coverage['bk'].str.startswith(puzzle_id)]
    if len(df_p) == 0:
        continue
    found = df_p['found_by_prob'].sum() + df_p['found_by_kg'].sum() > 0
    found_both = (df_p['found_by_prob'] & df_p['found_by_kg']).sum()
    print(f"  {puzzle_id}: {found} ({found_both} both) | ranks: {df_p['rank_a'].tolist()} / {df_p['rank_b'].tolist()}")

print("=" * 120)

EXPERIMENT 2: Pool Coverage Analysis

📊 Overall Statistics:
Total subkeys: 172
Average candidates per subkey: 14.0
Average unique candidates: 7.3
Average duplicate ratio: 0.378

📊 GT Found By:
  Probmul only:  0
  Kgmon only:    0
  Both:          0      False
1      False
2      False
3      False
4      False
       ...  
167    False
168    False
169    False
170    False
171    False
Name: found_by_prob, Length: 172, dtype: bool
  Neither:       0      True
1      True
2      True
3      True
4      True
       ... 
167    True
168    True
169    True
170    True
171    True
Name: found_by_prob, Length: 172, dtype: bool

📊 GT Rank Distribution (Probmul):
rank_a
1.0     36
2.0      9
3.0      6
5.0      3
10.0     1

📊 GT Rank Distribution (Kgmon):
rank_b
1.0    36
2.0    11
3.0     3
5.0     3
6.0     1
9.0     1

📌 Per-puzzle GT Presence:
  0934a4d8: False (0 both) | ranks: [nan] / [nan]
  135a2760: True (1 both) | ranks: [1.0] / [1.0]
  136b0064: False (0 both) | ranks: [nan] / [

In [21]:
# ================================================================
# EXPERIMENT 3: Why Scorer Misses GT
# Compare Rank1 vs GT Rank features (beam, aug, etc.)
# ================================================================

print("=" * 120)
print("EXPERIMENT 3: Why Scorer Misses GT (Rank1 vs GT)")
print("=" * 120)

miss_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank_a = find_rank(ranked_a, gt)
        
        if rank_a is None or rank_a <= 2:
            continue  # Skip if GT is already in top 2
        
        # Get Rank1 and GT candidates
        rank1 = ranked_a[0] if len(ranked_a) > 0 else None
        rank1_features = None
        gt_features = None
        
        for cand in g.values():
            if same_grid(cand["solution"], rank1):
                rank1_features = cand
            if same_grid(cand["solution"], gt):
                gt_features = cand
        
        if rank1_features is None or gt_features is None:
            continue
        
        miss_data.append({
            "bk": bk,
            "rank_a": rank_a,
            "rank1_beam": rank1_features["beam_score"],
            "gt_beam": gt_features["beam_score"],
            "rank1_aug_mean": np.mean(rank1_features["score_aug"]),
            "gt_aug_mean": np.mean(gt_features["score_aug"]),
            "rank1_aug_std": np.std(rank1_features["score_aug"]),
            "gt_aug_std": np.std(gt_features["score_aug"]),
        })

df_miss = pd.DataFrame(miss_data)

if len(df_miss) > 0:
    print(f"\n📊 Cases where GT rank > 2: {len(df_miss)}")
    print("\n📌 Detailed comparison:")
    print(df_miss[['bk', 'rank_a', 'rank1_beam', 'gt_beam', 'rank1_aug_mean', 'gt_aug_mean']].to_string(index=False))
    
    print("\n📊 Aggregate comparison:")
    print(f"  Beam score (Rank1): {df_miss['rank1_beam'].mean():.4f}")
    print(f"  Beam score (GT):    {df_miss['gt_beam'].mean():.4f}")
    print(f"  Aug mean (Rank1):   {df_miss['rank1_aug_mean'].mean():.4f}")
    print(f"  Aug mean (GT):      {df_miss['gt_aug_mean'].mean():.4f}")
    
    # Which metric is pushing GT down?
    df_miss['beam_advantage'] = df_miss['rank1_beam'] - df_miss['gt_beam']
    df_miss['aug_advantage'] = df_miss['rank1_aug_mean'] - df_miss['gt_aug_mean']
    
    print("\n📊 Metric Advantage (Rank1 - GT):")
    print(f"  (Negative means GT is better)")
    print(f"  Beam advantage mean: {df_miss['beam_advantage'].mean():.4f}")
    print(f"  Aug advantage mean:  {df_miss['aug_advantage'].mean():.4f}")
else:
    print("\n⚠️  No cases where GT rank > 2 found (or all GT are in top 2).")

print("=" * 120)

EXPERIMENT 3: Why Scorer Misses GT (Rank1 vs GT)

📊 Cases where GT rank > 2: 10

📌 Detailed comparison:
        bk  rank_a  rank1_beam  gt_beam  rank1_aug_mean  gt_aug_mean
b6f77b65_0       3    1.184737 1.314751        4.184703    12.782489
58490d8a_0       5    0.804758 1.518016        3.630549    12.231531
221dfab4_1       5    0.550267 0.549116        3.239385    16.591787
d35bdbdc_1       3    1.338922 0.004392        8.635913     7.947761
8b7bacbf_1      10    1.191346 1.308880        3.595679    12.755542
b5ca7ac4_0       3    0.491224 1.298475        6.599875     6.463332
2b83f449_0       3    1.387508 0.611699       15.248848    17.152351
3dc255db_0       5    1.524024 1.193745        5.582767    12.268042
6e453dd6_0       3    0.604871 0.005501        5.155205     6.603239
dfadab01_0       3    0.011880 0.327038        6.516096    11.598809

📊 Aggregate comparison:
  Beam score (Rank1): 0.9090
  Beam score (GT):    0.8132
  Aug mean (Rank1):   6.2389
  Aug mean (GT):      11.

In [22]:
# ================================================================
# EXPERIMENT 4: Agreement Statistics
# Distribution of (Probmul Rank, Kgmon Rank) for all GT
# ================================================================

print("=" * 120)
print("EXPERIMENT 4: Agreement Statistics (Probmul vs Kgmon)")
print("=" * 120)

agreement_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        rank_a = find_rank(ranked_a, gt)
        rank_b = find_rank(ranked_b, gt)
        
        agreement_data.append({
            "bk": bk,
            "rank_a": rank_a if rank_a is not None else 999,
            "rank_b": rank_b if rank_b is not None else 999,
            "found": rank_a is not None or rank_b is not None,
        })

df_agree = pd.DataFrame(agreement_data)

print("\n📊 Rank Pair Distribution (Probmul, Kgmon):")
pair_counts = df_agree[df_agree['found'] == True].groupby(['rank_a', 'rank_b']).size().reset_index(name='count')
if len(pair_counts) > 0:
    print(pair_counts.to_string(index=False))
else:
    print("  No GT found in any candidate pool.")

print("\n📊 Marginal Distributions:")
print("\nProbmul Rank:")
print(df_agree[df_agree['rank_a'] < 999]['rank_a'].value_counts().sort_index().to_string())
print("\nKgmon Rank:")
print(df_agree[df_agree['rank_b'] < 999]['rank_b'].value_counts().sort_index().to_string())

# Agreement: same rank
df_agree['same_rank'] = (df_agree['rank_a'] == df_agree['rank_b']) & (df_agree['rank_a'] < 999)
same_count = df_agree['same_rank'].sum()
total_found = df_agree['found'].sum()
print(f"\n📊 Agreement:")
print(f"  Same rank in both scorers: {same_count}/{total_found} ({same_count/total_found*100:.1f}%)")

# Correlation
if total_found > 1:
    corr = df_agree[df_agree['found'] == True][['rank_a', 'rank_b']].corr().iloc[0,1]
    print(f"  Rank correlation: {corr:.3f}")

print("=" * 120)

EXPERIMENT 4: Agreement Statistics (Probmul vs Kgmon)

📊 Rank Pair Distribution (Probmul, Kgmon):
 rank_a  rank_b  count
      1       1     35
      1       2      1
      2       1      1
      2       2      8
      3       2      2
      3       3      3
      3       5      1
      5       5      2
      5       6      1
     10       9      1

📊 Marginal Distributions:

Probmul Rank:
rank_a
1     36
2      9
3      6
5      3
10     1

Kgmon Rank:
rank_b
1    36
2    11
3     3
5     3
6     1
9     1

📊 Agreement:
  Same rank in both scorers: 48/55 (87.3%)
  Rank correlation: 0.961


In [23]:
# ================================================================
# EXPERIMENT 5: Candidate Survival Analysis
# Where does GT candidate get lost?
# ================================================================

print("=" * 120)
print("EXPERIMENT 5: Candidate Survival Analysis")
print("=" * 120)

survival_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        gt_hash = hashable(gt)
        
        # Track GT through pipeline
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        # Stage 1: Generated candidates
        total_generated = len(g)
        
        # Stage 2: Parsed/Deduplicated (by hash)
        unique_hashes = set()
        parsed_count = 0
        for cand in g.values():
            h = hashable(cand["solution"])
            if h not in unique_hashes:
                unique_hashes.add(h)
                parsed_count += 1
        
        # Stage 3: In ranked list
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        rank_a = find_rank(ranked_a, gt)
        rank_b = find_rank(ranked_b, gt)
        
        # Stage 4: Selected by current dual
        current_dual = decoder.run_dual_selection().get(bk, [])
        selected_by_dual = any(same_grid(c, gt) for c in current_dual)
        
        survival_data.append({
            "bk": bk,
            "generated": total_generated,
            "parsed": parsed_count,
            "rank_a": rank_a if rank_a is not None else "Not found",
            "rank_b": rank_b if rank_b is not None else "Not found",
            "selected_by_dual": selected_by_dual,
        })

df_survival = pd.DataFrame(survival_data)

print("\n📊 Per-subkey Survival Summary:")
print(df_survival.to_string(index=False))

print("\n📊 Aggregate Survival Statistics:")
total_subkeys = len(df_survival)
gt_found_in_ranked = ((df_survival['rank_a'] != "Not found") | (df_survival['rank_b'] != "Not found")).sum()
selected = df_survival['selected_by_dual'].sum()

print(f"  Total subkeys: {total_subkeys}")
print(f"  GT found in ranked list: {gt_found_in_ranked} ({gt_found_in_ranked/total_subkeys*100:.1f}%)")
print(f"  GT selected by dual: {selected} ({selected/total_subkeys*100:.1f}%)")

print("\n📌 Where GT is lost:")
lost_at_ranking = total_subkeys - gt_found_in_ranked
lost_at_selection = gt_found_in_ranked - selected
print(f"  Lost at ranking stage: {lost_at_ranking} ({lost_at_ranking/total_subkeys*100:.1f}%)")
print(f"  Lost at selection stage: {lost_at_selection} ({lost_at_selection/total_subkeys*100:.1f}%)")

# Per-puzzle breakdown
print("\n📌 Per-puzzle breakdown:")
for puzzle_id in sorted(ALL_BKS.keys()):
    df_p = df_survival[df_survival['bk'].str.startswith(puzzle_id)]
    if len(df_p) == 0:
        continue
    found = ((df_p['rank_a'] != "Not found") | (df_p['rank_b'] != "Not found")).sum()
    selected = df_p['selected_by_dual'].sum()
    print(f"  {puzzle_id}: {found}/{len(df_p)} found, {selected}/{len(df_p)} selected")

print("=" * 120)

EXPERIMENT 5: Candidate Survival Analysis

📊 Per-subkey Survival Summary:
        bk  generated  parsed    rank_a    rank_b  selected_by_dual
b99e7126_0         18       6 Not found Not found             False
4c7dc4dd_1         17      14 Not found Not found             False
4c7dc4dd_0         22       7 Not found Not found             False
88bcf3b4_0         16      15 Not found Not found             False
88bcf3b4_1          7       5 Not found Not found             False
1ae2feb7_0         12       9 Not found Not found             False
1ae2feb7_1         17       3 Not found Not found             False
1ae2feb7_2         23      14         1         1              True
d8e07eb2_1         26      10         1         1              True
d8e07eb2_0         17       3 Not found Not found             False
78332cb0_1         20      10         1         1              True
78332cb0_0         21      12 Not found Not found             False
89565ca0_0         22      16 Not found No

In [24]:
# ================================================================
# EXPERIMENT 6: Beam Search Analysis (142ca369) - FIXED
# ================================================================

print("=" * 120)
print("EXPERIMENT 6: Beam Search Analysis (142ca369)")
print("=" * 120)

target_puzzle = "142ca369"

if target_puzzle not in ALL_BKS:
    print(f"⚠️  {target_puzzle} not found in decoded_results")
else:
    for bk in ALL_BKS[target_puzzle]:
        if target_puzzle not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[target_puzzle]):
            continue
        
        gt = GT[target_puzzle][sub_idx]
        gt_arr = np.asarray(gt)  # Convert GT to numpy array once
        
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        
        print(f"\n📌 {bk}")
        print(f"   Total candidates: {len(g)}")
        
        # Find closest candidate (by similarity)
        best_sim = -1
        best_cand = None
        best_rank = None
        best_features = None
        
        for cand in g.values():
            cand_arr = np.asarray(cand["solution"])
            if cand_arr.shape == gt_arr.shape:
                sim = np.mean(cand_arr == gt_arr) * 100
            else:
                sim = 0
            if sim > best_sim:
                best_sim = sim
                best_cand = cand
                best_rank = find_rank(ranked_a, cand["solution"])
                best_features = cand
        
        print(f"   Closest candidate similarity: {best_sim:.2f}%")
        print(f"   Closest candidate rank: {best_rank}")
        if best_features:
            print(f"   Closest beam score: {best_features['beam_score']:.4f}")
            print(f"   Closest aug mean: {np.mean(best_features['score_aug']):.4f}")
        
        # Show differences
        if best_cand is not None and best_sim < 100:
            cand_arr = np.asarray(best_cand["solution"])
            diff_coords = np.argwhere(cand_arr != gt_arr)
            print(f"\n   Differing cells ({len(diff_coords)} total):")
            for idx, coord in enumerate(diff_coords[:10]):
                print(f"      {coord}: GT={gt_arr[coord[0], coord[1]]}, Cand={cand_arr[coord[0], coord[1]]}")
            if len(diff_coords) > 10:
                print(f"      ... and {len(diff_coords) - 10} more")
        
        # Show top 5 candidates with their features
        print(f"\n   Top 5 candidates (Probmul) with features:")
        for i, cand in enumerate(ranked_a[:5]):
            is_gt = same_grid(cand, gt)
            features = None
            for c in g.values():
                if same_grid(c["solution"], cand):
                    features = c
                    break
            if features:
                print(f"      Rank {i+1}: beam={features['beam_score']:.4f}, aug_mean={np.mean(features['score_aug']):.4f}, is_GT={is_gt}")
            else:
                print(f"      Rank {i+1}: (features not found)")

print("=" * 120)

EXPERIMENT 6: Beam Search Analysis (142ca369)

📌 142ca369_0
   Total candidates: 22
   Closest candidate similarity: 72.84%
   Closest candidate rank: 3
   Closest beam score: 1.2060
   Closest aug mean: 17.9438

   Differing cells (88 total):
      [2 9]: GT=2, Cand=0
      [ 2 11]: GT=8, Cand=0
      [ 2 13]: GT=6, Cand=0
      [ 2 15]: GT=4, Cand=0
      [3 8]: GT=2, Cand=0
      [ 3 10]: GT=8, Cand=0
      [ 3 12]: GT=6, Cand=0
      [ 3 14]: GT=4, Cand=0
      [4 7]: GT=2, Cand=0
      [4 9]: GT=8, Cand=0
      ... and 78 more

   Top 5 candidates (Probmul) with features:
      Rank 1: beam=1.0941, aug_mean=12.0258, is_GT=False
      Rank 2: beam=0.1978, aug_mean=14.6497, is_GT=False
      Rank 3: beam=1.2060, aug_mean=17.9438, is_GT=False
      Rank 4: beam=1.0052, aug_mean=19.3059, is_GT=False
      Rank 5: beam=1.5052, aug_mean=20.8152, is_GT=False

📌 142ca369_1
   Total candidates: 26
   Closest candidate similarity: 93.16%
   Closest candidate rank: 9
   Closest beam score: 1

In [25]:
# ================================================================
# EXPERIMENT 7: Missing Decoded Results
# Investigate why 62593bfd and 8b7bacbf are missing
# ================================================================

print("=" * 120)
print("EXPERIMENT 7: Missing Decoded Results Analysis")
print("=" * 120)

missing_puzzles = ["62593bfd", "8b7bacbf"]

# Check if they exist in the original dataset
try:
    data
except NameError:
    # Load data if not available
    from arc_loader import ArcDataset
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    data = ArcDataset.from_file(test_path)

all_puzzle_ids = set(data.queries.keys())

for pid in missing_puzzles:
    print(f"\n📌 {pid}:")
    
    # Check if in dataset
    if pid in all_puzzle_ids:
        print(f"   ✅ Exists in original dataset")
        puzzle_data = data.queries[pid]
        print(f"   Number of train examples: {len(puzzle_data['train'])}")
        print(f"   Number of test queries: {len(puzzle_data['test'])}")
    else:
        print(f"   ❌ Does NOT exist in original dataset")
        continue
    
    # Check if any subkey exists in decoded_results
    matching = [k for k in decoder.decoded_results.keys() if k.startswith(pid)]
    if matching:
        print(f"   ✅ Found in decoded_results: {matching}")
    else:
        print(f"   ❌ NOT found in decoded_results")
        
        # Check if it was queued (by checking starter.py DEBUG_PUZZLES)
        print(f"   🔍 Possible causes:")
        print(f"      - Not in DEBUG_PUZZLES list (if RUN_FULL_EVAL=False)")
        print(f"      - Worker timeout before processing")
        print(f"      - Worker crash during processing")
        print(f"      - Fallback save overwritten?")
        
        # Check if there are any fallback files
        import os
        fallback_path = f"/kaggle/inference_outputs/{pid}_0.fallback"
        if os.path.exists(fallback_path):
            print(f"      - Fallback file exists: {fallback_path}")
        else:
            print(f"      - No fallback file found")

print("\n📊 Overall decoded_results coverage:")
all_expected = set()
for pid in all_puzzle_ids:
    # Count test queries
    num_tests = len(data.queries[pid]['test'])
    for i in range(num_tests):
        all_expected.add(f"{pid}_{i}")

decoded_set = set(decoder.decoded_results.keys())
missing = all_expected - decoded_set

print(f"   Total expected subkeys: {len(all_expected)}")
print(f"   Total decoded subkeys: {len(decoded_set)}")
print(f"   Missing subkeys: {len(missing)}")

if len(missing) > 0:
    # Group missing by puzzle
    missing_by_puzzle = defaultdict(list)
    for bk in missing:
        pid = bk.rsplit("_", 1)[0]
        missing_by_puzzle[pid].append(bk)
    
    print("\n   Missing by puzzle:")
    for pid, bks in sorted(missing_by_puzzle.items()):
        print(f"      {pid}: {bks}")

print("=" * 120)

EXPERIMENT 7: Missing Decoded Results Analysis

📌 62593bfd:
   ✅ Exists in original dataset
   Number of train examples: 2
   Number of test queries: 2
   ✅ Found in decoded_results: ['62593bfd_1', '62593bfd_0']

📌 8b7bacbf:
   ✅ Exists in original dataset
   Number of train examples: 4
   Number of test queries: 2
   ✅ Found in decoded_results: ['8b7bacbf_1', '8b7bacbf_0']

📊 Overall decoded_results coverage:
   Total expected subkeys: 172
   Total decoded subkeys: 172
   Missing subkeys: 0


In [26]:
# ================================================================
# EXPERIMENT 8: Score Function Audit (GT vs Rank1)
# ================================================================

print("=" * 120)
print("EXPERIMENT 8: Score Function Audit (GT vs Rank1)")
print("=" * 120)

audit_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank_a = find_rank(ranked_a, gt)
        ranked_b = score_kgmon(g)
        rank_b = find_rank(ranked_b, gt)
        
        if rank_a is None or rank_a <= 2:
            continue  # শুধু মিস হওয়া কেসগুলো দেখি
        
        # Rank1 ও GT খুঁজে বের করি
        rank1_cand = ranked_a[0] if len(ranked_a) > 0 else None
        rank1_features = None
        gt_features = None
        
        for cand in g.values():
            if same_grid(cand["solution"], rank1_cand):
                rank1_features = cand
            if same_grid(cand["solution"], gt):
                gt_features = cand
        
        if rank1_features is None or gt_features is None:
            continue
        
        # বিস্তারিত প্রিন্ট
        print(f"\n📌 {bk} (GT Rank: {rank_a} / {rank_b})")
        print("-" * 80)
        
        print("  Rank1 Candidate:")
        print(f"    beam_score   : {rank1_features['beam_score']:.6f}")
        print(f"    score_aug    : {rank1_features['score_aug']}")
        print(f"    aug_mean     : {np.mean(rank1_features['score_aug']):.6f}")
        print(f"    aug_std      : {np.std(rank1_features['score_aug']):.6f}")
        
        print("\n  GT Candidate:")
        print(f"    beam_score   : {gt_features['beam_score']:.6f}")
        print(f"    score_aug    : {gt_features['score_aug']}")
        print(f"    aug_mean     : {np.mean(gt_features['score_aug']):.6f}")
        print(f"    aug_std      : {np.std(gt_features['score_aug']):.6f}")
        
        # getter_full_probmul_3-এর কম্পোনেন্ট বের করি (শুধু এই ক্যান্ডিডেটের জন্য নয়, পুরো পুলের জন্য)
        # কিন্তু আমরা ব্যাখ্যা করি কেন GT পিছিয়ে আছে
        print("\n  🔍 Analysis:")
        beam_diff = rank1_features['beam_score'] - gt_features['beam_score']
        aug_diff = np.mean(rank1_features['score_aug']) - np.mean(gt_features['score_aug'])
        
        if beam_diff < 0:
            print(f"    ✅ GT has BETTER beam score (lower is better): {beam_diff:.4f}")
        else:
            print(f"    ❌ GT has WORSE beam score: {beam_diff:.4f}")
            
        if aug_diff < 0:
            print(f"    ✅ GT has BETTER aug score (lower is better): {aug_diff:.4f}")
        else:
            print(f"    ❌ GT has WORSE aug score: {aug_diff:.4f}")
        
        # সংরক্ষণ
        audit_data.append({
            "bk": bk,
            "rank_a": rank_a,
            "rank_b": rank_b,
            "rank1_beam": rank1_features['beam_score'],
            "gt_beam": gt_features['beam_score'],
            "rank1_aug_mean": np.mean(rank1_features['score_aug']),
            "gt_aug_mean": np.mean(gt_features['score_aug']),
        })

print("\n" + "=" * 120)
print("✅ Experiment 8 Complete.")
print("=" * 120)

EXPERIMENT 8: Score Function Audit (GT vs Rank1)

📌 b6f77b65_0 (GT Rank: 3 / 3)
--------------------------------------------------------------------------------
  Rank1 Candidate:
    beam_score   : 1.184737
    score_aug    : [5.302924156188965, 7.02697229385376, 0.9967935085296631, 8.624067306518555, 2.6326370239257812, 0.004034624435007572, 8.553853988647461, 0.3363427519798279]
    aug_mean     : 4.184703
    aug_std      : 3.409287

  GT Candidate:
    beam_score   : 1.314751
    score_aug    : [25.009023666381836, 14.759342193603516, 19.129575729370117, 0.3466006815433502, 0.8117967844009399, 15.518550872802734, 1.7527846097946167, 24.932233810424805]
    aug_mean     : 12.782489
    aug_std      : 9.800428

  🔍 Analysis:
    ✅ GT has BETTER beam score (lower is better): -0.1300
    ✅ GT has BETTER aug score (lower is better): -8.5978

📌 58490d8a_0 (GT Rank: 5 / 6)
--------------------------------------------------------------------------------
  Rank1 Candidate:
    beam_score  

In [27]:
# ================================================================
# EXPERIMENT 9: Score Function Weight Analysis
# ================================================================

print("=" * 120)
print("EXPERIMENT 9: Score Function Weight Analysis")
print("=" * 120)

def analyze_score_weights(guesses_dict):
    """
    score_full_probmul_3-এর জন্য beam ও aug-এর কনট্রিবিউশন বের করে
    """
    guess_list = list(guesses_dict.values())
    if not guess_list:
        return None
    
    beam_scores = [g["beam_score"] for g in guess_list]
    baseline = max(beam_scores) + 1.0
    
    # প্রতিটি ক্যান্ডিডেটের জন্য বিস্তারিত
    details = []
    for g in guess_list:
        beam_contrib = baseline - g["beam_score"]
        aug_contrib = np.sum([baseline - s for s in g["score_aug"]])
        total = beam_contrib + aug_contrib
        
        details.append({
            "beam": g["beam_score"],
            "beam_contrib": beam_contrib,
            "aug_mean": np.mean(g["score_aug"]),
            "aug_contrib": aug_contrib,
            "total": total
        })
    
    return {
        "baseline": baseline,
        "details": details,
        "avg_beam_contrib": np.mean([d["beam_contrib"] for d in details]),
        "avg_aug_contrib": np.mean([d["aug_contrib"] for d in details]),
    }

print("\n📊 Analyzing score_full_probmul_3 components...")

weight_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank_a = find_rank(ranked_a, gt)
        
        if rank_a is None:
            continue
        
        analysis = analyze_score_weights(g)
        if analysis is None:
            continue
        
        # GT-র জন্য আলাদা
        gt_features = None
        for cand in g.values():
            if same_grid(cand["solution"], gt):
                gt_features = cand
                break
        
        if gt_features is None:
            continue
        
        gt_beam_contrib = analysis["baseline"] - gt_features["beam_score"]
        gt_aug_contrib = np.sum([analysis["baseline"] - s for s in gt_features["score_aug"]])
        gt_total = gt_beam_contrib + gt_aug_contrib
        
        weight_data.append({
            "bk": bk,
            "rank_a": rank_a,
            "baseline": analysis["baseline"],
            "gt_beam_contrib": gt_beam_contrib,
            "gt_aug_contrib": gt_aug_contrib,
            "gt_total": gt_total,
            "avg_beam_contrib": analysis["avg_beam_contrib"],
            "avg_aug_contrib": analysis["avg_aug_contrib"],
        })

df_weights = pd.DataFrame(weight_data)

if len(df_weights) > 0:
    print(f"\n📊 Analyzed {len(df_weights)} subkeys where GT exists:")
    print(df_weights[['bk', 'rank_a', 'gt_beam_contrib', 'gt_aug_contrib', 'gt_total']].to_string(index=False))
    
    print("\n📊 Aggregate Contributions (GT candidates):")
    print(f"  Average beam contribution:  {df_weights['gt_beam_contrib'].mean():.4f}")
    print(f"  Average aug contribution:   {df_weights['gt_aug_contrib'].mean():.4f}")
    print(f"  Beam % of total:            {df_weights['gt_beam_contrib'].mean() / df_weights['gt_total'].mean() * 100:.2f}%")
    print(f"  Aug % of total:             {df_weights['gt_aug_contrib'].mean() / df_weights['gt_total'].mean() * 100:.2f}%")
    
    print("\n📊 Aggregate Contributions (Average over ALL candidates):")
    print(f"  Average beam contribution:  {df_weights['avg_beam_contrib'].mean():.4f}")
    print(f"  Average aug contribution:   {df_weights['avg_aug_contrib'].mean():.4f}")
    print(f"  Beam % of total:            {df_weights['avg_beam_contrib'].mean() / (df_weights['avg_beam_contrib'].mean() + df_weights['avg_aug_contrib'].mean()) * 100:.2f}%")
    print(f"  Aug % of total:             {df_weights['avg_aug_contrib'].mean() / (df_weights['avg_beam_contrib'].mean() + df_weights['avg_aug_contrib'].mean()) * 100:.2f}%")
else:
    print("⚠️  No data to analyze.")

print("\n" + "=" * 120)
print("✅ Experiment 9 Complete.")
print("=" * 120)

EXPERIMENT 9: Score Function Weight Analysis

📊 Analyzing score_full_probmul_3 components...

📊 Analyzed 55 subkeys where GT exists:
        bk  rank_a  gt_beam_contrib  gt_aug_contrib    gt_total
1ae2feb7_2       1         2.588619       -2.872188   -0.283569
d8e07eb2_1       1         1.316240        8.484659    9.800899
78332cb0_1       1         2.027397      -27.879909  -25.852512
e8686506_0       1         2.497420      -16.942734  -14.445314
2d0172a1_0       1         1.534750      -11.171698   -9.636948
2d0172a1_1       2         2.008633      -11.025070   -9.016437
38007db0_0       1         1.202286        6.533462    7.735748
38007db0_1       1         1.001601        8.013134    9.014736
45a5af55_0       2         1.772660      -10.810586   -9.037927
7ed72f31_0       1         1.969191      -14.822067  -12.852876
7c66cb00_0       1         2.332931       11.719091   14.052022
71e489b6_0       1         2.382038      -27.050902  -24.668864
71e489b6_1       1         2.174008

In [28]:
# ================================================================
# EXPERIMENT 10: Scoring Optimization (Highest ROI)
# ================================================================

print("=" * 120)
print("EXPERIMENT 10: Scoring Optimization")
print("=" * 120)

def custom_getter(guesses, w_beam=1.0, w_aug=1.0, use_baseline=True):
    """
    Custom scoring function with adjustable weights for beam and aug.
    w_beam: weight for beam score contribution
    w_aug: weight for augmentation score contribution
    """
    if not guesses:
        return 0
    
    beam_scores = [g["beam_score"] for g in guesses]
    if use_baseline:
        baseline = max(beam_scores) + 1.0
        beam_contrib = np.sum([baseline - g["beam_score"] for g in guesses])
        aug_contrib = np.mean([np.sum([baseline - s for s in g["score_aug"]]) for g in guesses])
    else:
        # সরাসরি (lower is better)
        beam_contrib = -np.mean(beam_scores)
        aug_contrib = -np.mean([np.mean(g["score_aug"]) for g in guesses])
    
    return w_beam * beam_contrib + w_aug * aug_contrib

def test_scoring_variant(w_beam, w_aug, use_baseline=True, name="Custom"):
    """একটি নির্দিষ্ট ওয়েট কম্বিনেশন টেস্ট করে"""
    total_score = 0.0
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1
    
    correct_count = 0
    total_puzzles = len(decoder.decoded_results)
    
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        
        g = {k: v for k, v in guesses_dict.items()}
        
        # Custom scoring এর মাধ্যমে র‍্যাঙ্কিং
        scored_list = []
        for cand in g.values():
            score = custom_getter([cand], w_beam, w_aug, use_baseline)
            scored_list.append((score, cand["solution"]))
        
        # সাজানো (descending, কারণ বেশি স্কোর ভালো)
        scored_list.sort(key=lambda x: x[0], reverse=True)
        ranked_custom = [s[1] for s in scored_list]
        
        rank = find_rank(ranked_custom, correct)
        
        # Attempt1 = Top1, Attempt2 = Top2 (যদি Rank 1 হয়, তাহলে Attempt1-এ সল্ভ)
        solved = False
        if rank == 1:
            solved = True
        elif rank == 2 and len(ranked_custom) > 1:
            solved = True
        
        if solved:
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            correct_count += 1
    
    return total_score, correct_count, total_puzzles

# বেসলাইন (বর্তমান)
current_dual = decoder.run_dual_selection()
base_score = 0.0
num_tasks_per_puzzle = defaultdict(int)
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    num_tasks_per_puzzle[puzzle_id] += 1

for bk, picks in current_dual.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue
    if any(np.array_equal(p, correct) for p in picks):
        puzzle_id = bk.rsplit("_", 1)[0]
        base_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

print(f"\n📊 Baseline (Current Dual): {base_score:.2f}")

# বিভিন্ন ওয়েট কম্বিনেশন টেস্ট
variants = [
    ("w_beam=1.0, w_aug=0.0 (Beam only)", 1.0, 0.0, True),
    ("w_beam=0.0, w_aug=1.0 (Aug only)", 0.0, 1.0, True),
    ("w_beam=0.8, w_aug=0.2", 0.8, 0.2, True),
    ("w_beam=0.5, w_aug=0.5", 0.5, 0.5, True),
    ("w_beam=0.2, w_aug=0.8", 0.2, 0.8, True),
    ("w_beam=1.0, w_aug=0.0 (No baseline, direct)", 1.0, 0.0, False),
    ("w_beam=0.0, w_aug=1.0 (No baseline, direct)", 0.0, 1.0, False),
]

results = []
print("\n📊 Testing weight variants...")
for name, wb, wa, use_bl in variants:
    score, correct, total = test_scoring_variant(wb, wa, use_bl, name)
    change = score - base_score
    results.append((name, score, correct, total, change))
    print(f"  {name:<40} | Score: {score:5.2f} | Solved: {correct:2}/{total} | Change: {change:+.2f}")

# বেস্ট খুঁজি
best = max(results, key=lambda x: x[1])
print("\n" + "=" * 120)
print(f"🏆 WINNER: {best[0]} with score {best[1]:.2f}")
print("=" * 120)

print("\n📌 RECOMMENDATION:")
print(f"  If '{best[0]}' beats baseline, we should modify score_full_probmul_3()")
print("  to use this weight combination in the actual code.")
print("  This is a ONE-LINE change in arc_decoder.py (getter_full_probmul_3).")

print("\n" + "=" * 120)
print("✅ Experiment 10 Complete.")
print("=" * 120)

EXPERIMENT 10: Scoring Optimization

📊 Baseline (Current Dual): 33.50

📊 Testing weight variants...
  w_beam=1.0, w_aug=0.0 (Beam only)        | Score: 24.17 | Solved: 33/172 | Change: -9.33
  w_beam=0.0, w_aug=1.0 (Aug only)         | Score: 26.50 | Solved: 37/172 | Change: -7.00
  w_beam=0.8, w_aug=0.2                    | Score: 26.50 | Solved: 37/172 | Change: -7.00
  w_beam=0.5, w_aug=0.5                    | Score: 26.50 | Solved: 37/172 | Change: -7.00
  w_beam=0.2, w_aug=0.8                    | Score: 26.50 | Solved: 37/172 | Change: -7.00
  w_beam=1.0, w_aug=0.0 (No baseline, direct) | Score: 24.33 | Solved: 32/172 | Change: -9.17
  w_beam=0.0, w_aug=1.0 (No baseline, direct) | Score: 28.83 | Solved: 40/172 | Change: -4.67

🏆 WINNER: w_beam=0.0, w_aug=1.0 (No baseline, direct) with score 28.83

📌 RECOMMENDATION:
  If 'w_beam=0.0, w_aug=1.0 (No baseline, direct)' beats baseline, we should modify score_full_probmul_3()
  to use this weight combination in the actual code.
  This

In [29]:
# ================================================================
# EXPERIMENT 11: Candidate Feature Correlation
# ================================================================

print("=" * 120)
print("EXPERIMENT 11: Candidate Feature Correlation")
print("=" * 120)

correlation_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        
        for cand in g.values():
            cand_arr = cand["solution"]
            is_gt = same_grid(cand_arr, gt)
            rank = find_rank(ranked_a, cand_arr)
            
            if rank is not None:
                correlation_data.append({
                    "bk": bk,
                    "rank": rank,
                    "beam": cand["beam_score"],
                    "aug_mean": np.mean(cand["score_aug"]),
                    "is_gt": is_gt,
                })

df_corr = pd.DataFrame(correlation_data)

if len(df_corr) > 0:
    print(f"\n📊 Total candidates analyzed: {len(df_corr)}")
    print(f"   GT candidates: {df_corr['is_gt'].sum()}")
    
    print("\n📊 Correlation with Rank:")
    corr_beam = df_corr['rank'].corr(df_corr['beam'])
    corr_aug = df_corr['rank'].corr(df_corr['aug_mean'])
    print(f"   corr(rank, beam)     : {corr_beam:.4f}")
    print(f"   corr(rank, aug_mean) : {corr_aug:.4f}")
    
    print("\n📊 Feature Statistics:")
    print(df_corr[['rank', 'beam', 'aug_mean']].describe().to_string())
    
    # GT candidates feature comparison
    df_gt = df_corr[df_corr['is_gt'] == True]
    df_non_gt = df_corr[df_corr['is_gt'] == False]
    
    if len(df_gt) > 0:
        print("\n📊 GT vs Non-GT Feature Comparison:")
        print(f"  GT beam mean:       {df_gt['beam'].mean():.4f}")
        print(f"  Non-GT beam mean:   {df_non_gt['beam'].mean():.4f}")
        print(f"  GT aug mean:        {df_gt['aug_mean'].mean():.4f}")
        print(f"  Non-GT aug mean:    {df_non_gt['aug_mean'].mean():.4f}")
else:
    print("⚠️  No data available.")

print("=" * 120)


# ================================================================
# EXPERIMENT 12: Rank Drift (Probmul vs Kgmon)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 12: Rank Drift (Probmul vs Kgmon)")
print("=" * 120)

drift_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        # Each candidate's rank in both systems
        for cand in g.values():
            cand_arr = cand["solution"]
            rank_a = find_rank(ranked_a, cand_arr)
            rank_b = find_rank(ranked_b, cand_arr)
            
            if rank_a is not None and rank_b is not None:
                drift_data.append({
                    "bk": bk,
                    "rank_a": rank_a,
                    "rank_b": rank_b,
                    "drift": abs(rank_a - rank_b),
                    "is_gt": same_grid(cand_arr, gt),
                })

df_drift = pd.DataFrame(drift_data)

if len(df_drift) > 0:
    print(f"\n📊 Total candidates tracked: {len(df_drift)}")
    print(f"   GT candidates: {df_drift['is_gt'].sum()}")
    
    print("\n📊 Rank Drift Statistics:")
    print(f"   Mean drift:      {df_drift['drift'].mean():.2f}")
    print(f"   Median drift:    {df_drift['drift'].median():.2f}")
    print(f"   Max drift:       {df_drift['drift'].max()}")
    print(f"   Std drift:       {df_drift['drift'].std():.2f}")
    
    print("\n📊 Drift Distribution:")
    print(df_drift['drift'].value_counts().sort_index().to_string())
    
    print("\n📊 GT Candidates Drift:")
    df_gt_drift = df_drift[df_drift['is_gt'] == True]
    if len(df_gt_drift) > 0:
        print(df_gt_drift[['bk', 'rank_a', 'rank_b', 'drift']].to_string(index=False))
else:
    print("⚠️  No data available.")

print("=" * 120)


# ================================================================
# EXPERIMENT 13: Oracle Selection (MOST IMPORTANT)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 13: Oracle Selection (Theoretical Upper Bound)")
print("=" * 120)

def compute_oracle_score():
    """
    Oracle Selection: Always picks the GT if it exists in the candidate pool.
    This gives us the theoretical maximum score of any selection policy.
    """
    total_score = 0.0
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1
    
    solved_count = 0
    total_puzzles = len(decoder.decoded_results)
    
    for bk, guesses_dict in decoder.decoded_results.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        
        g = {k: v for k, v in guesses_dict.items()}
        ranked_a = score_full_probmul_3(g)
        
        # Oracle: GT যদি Rank 1-2-এ থাকে, তাহলে সল্ভ
        rank = find_rank(ranked_a, correct)
        
        if rank is not None and rank <= 2:
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            solved_count += 1
        elif rank is not None and rank > 2:
            # Oracle Attempt 2 = GT (if GT exists anywhere)
            puzzle_id = bk.rsplit("_", 1)[0]
            total_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
            solved_count += 1
    
    return total_score, solved_count, total_puzzles

# Current Dual Score
current_dual = decoder.run_dual_selection()
current_score = 0.0
num_tasks_per_puzzle = defaultdict(int)
for bk in decoder.decoded_results.keys():
    puzzle_id = bk.rsplit("_", 1)[0]
    num_tasks_per_puzzle[puzzle_id] += 1

for bk, picks in current_dual.items():
    correct = decoder.dataset.replies.get(bk, [None])[0]
    if correct is None:
        continue
    if any(np.array_equal(p, correct) for p in picks):
        puzzle_id = bk.rsplit("_", 1)[0]
        current_score += 1.0 / num_tasks_per_puzzle[puzzle_id]

# Oracle Score
oracle_score, oracle_solved, total = compute_oracle_score()

print(f"\n📊 Current Dual Score:   {current_score:.2f} ({current_score} puzzles)")
print(f"📊 Oracle Score:         {oracle_score:.2f} ({oracle_solved}/{total} puzzles)")
print(f"📊 Theoretical Gain:     {oracle_score - current_score:.2f}")
print(f"📊 Maximum Possible:     {oracle_score:.2f}")

print("\n📌 Interpretation:")
if oracle_score - current_score <= 2:
    print("   ✅ Selection is already near-optimal. Focus on GENERATION improvement.")
elif oracle_score - current_score <= 5:
    print("   ⚠️  Selection can improve modestly. Both selection and generation matter.")
else:
    print("   🚨 Selection has significant room for improvement. Focus on SELECTION redesign.")

print("=" * 120)


# ================================================================
# EXPERIMENT 14: Top-k Coverage
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 14: Top-k Coverage (Where is GT?)")
print("=" * 120)

coverage = {1: 0, 2: 0, 3: 0, 5: 0, 10: 0, 20: 0, 100: 0}
total_gt_found = 0
total_subkeys = 0

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        total_subkeys += 1
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank = find_rank(ranked_a, gt)
        
        if rank is not None:
            total_gt_found += 1
            for k in coverage.keys():
                if rank <= k:
                    coverage[k] += 1

print(f"\n📊 Total subkeys: {total_subkeys}")
print(f"📊 GT found in pool: {total_gt_found} ({total_gt_found/total_subkeys*100:.1f}%)")

print("\n📊 GT Coverage by Top-k:")
for k, count in sorted(coverage.items()):
    pct = count / total_subkeys * 100 if total_subkeys > 0 else 0
    print(f"   Top-{k:3d}: {count:2d}/{total_subkeys} ({pct:.1f}%)")

# Coverage improvement from Top1 to Top5
top1 = coverage[1]
top5 = coverage[5]
print(f"\n📊 Improvement Top1 → Top5: {top1} → {top5} (+{top5 - top1})")
print(f"   (Selection policy can recover at most {top5 - top1} additional puzzles)")

print("\n" + "=" * 120)
print("✅ All Experiments Complete.")
print("=" * 120)

EXPERIMENT 11: Candidate Feature Correlation

📊 Total candidates analyzed: 2416
   GT candidates: 398

📊 Correlation with Rank:
   corr(rank, beam)     : -0.0624
   corr(rank, aug_mean) : 0.0173

📊 Feature Statistics:
              rank         beam     aug_mean
count  2416.000000  2416.000000  2416.000000
mean      4.030215     6.519812    16.808163
std       3.562910    75.787921    75.553267
min       1.000000     0.000015     0.000225
25%       1.000000     0.203092     3.846232
50%       3.000000     0.742617     9.367046
75%       6.000000     1.234866    16.022229
max      19.000000   999.000000   999.000000

📊 GT vs Non-GT Feature Comparison:
  GT beam mean:       0.2822
  Non-GT beam mean:   7.7500
  GT aug mean:        2.7699
  Non-GT aug mean:    19.5769

EXPERIMENT 12: Rank Drift (Probmul vs Kgmon)

📊 Total candidates tracked: 2416
   GT candidates: 398

📊 Rank Drift Statistics:
   Mean drift:      0.30
   Median drift:    0.00
   Max drift:       5
   Std drift:       0.64

In [30]:
# ================================================================
# EXPERIMENT 15: Oracle Top5 Pattern Mining (⭐⭐⭐⭐⭐)
# ================================================================

print("=" * 120)
print("EXPERIMENT 15: Oracle Top5 Pattern Mining")
print("=" * 120)

pattern_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank_gt = find_rank(ranked_a, gt)
        
        # শুধু যেখানে GT Rank 1-5-এর মধ্যে আছে (Oracle ক্যাপচার করতে পারে)
        if rank_gt is None or rank_gt > 5:
            continue
        
        print(f"\n📌 {bk} (GT Rank: {rank_gt})")
        print("-" * 80)
        print(f"{'Rank':<6} {'Beam':<12} {'Aug Mean':<12} {'Aug Std':<12} {'Is GT':<8}")
        print("-" * 80)
        
        for i, cand in enumerate(ranked_a[:5], start=1):
            # Find features for this candidate
            features = None
            for c in g.values():
                if same_grid(c["solution"], cand):
                    features = c
                    break
            if features is None:
                continue
            
            is_gt = same_grid(cand, gt)
            aug_std = np.std(features["score_aug"])
            
            # Store for pattern analysis
            pattern_data.append({
                "bk": bk,
                "rank": i,
                "is_gt": is_gt,
                "beam": features["beam_score"],
                "aug_mean": np.mean(features["score_aug"]),
                "aug_std": aug_std,
            })
            
            mark = " <== GT" if is_gt else ""
            print(f"{i:<6} {features['beam_score']:<12.4f} {np.mean(features['score_aug']):<12.4f} {aug_std:<12.4f} {str(is_gt):<8}{mark}")

# Pattern analysis
df_pattern = pd.DataFrame(pattern_data)
if len(df_pattern) > 0:
    print("\n" + "=" * 80)
    print("📊 PATTERN ANALYSIS (GT vs Non-GT in Top5):")
    print("=" * 80)
    df_gt_top5 = df_pattern[df_pattern['is_gt'] == True]
    df_non_gt_top5 = df_pattern[df_pattern['is_gt'] == False]
    
    print("\n🔍 Feature Comparison in Top5:")
    print(f"  GT beam mean:     {df_gt_top5['beam'].mean():.4f}  |  Non-GT beam mean: {df_non_gt_top5['beam'].mean():.4f}")
    print(f"  GT aug_mean mean: {df_gt_top5['aug_mean'].mean():.4f}  |  Non-GT aug_mean: {df_non_gt_top5['aug_mean'].mean():.4f}")
    print(f"  GT aug_std mean:  {df_gt_top5['aug_std'].mean():.4f}  |  Non-GT aug_std:  {df_non_gt_top5['aug_std'].mean():.4f}")
    
    print("\n📌 Key Pattern:")
    if df_gt_top5['aug_mean'].mean() < df_non_gt_top5['aug_mean'].mean():
        print("  ✅ GT has LOWER aug_mean (better) than non-GT in Top5.")
    else:
        print("  ❌ GT has HIGHER aug_mean (worse) than non-GT in Top5.")
    
    if df_gt_top5['beam'].mean() < df_non_gt_top5['beam'].mean():
        print("  ✅ GT has LOWER beam (better) than non-GT in Top5.")
    else:
        print("  ❌ GT has HIGHER beam (worse) than non-GT in Top5.")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 16: Meta Selector (Logistic Regression) ⭐⭐⭐⭐⭐
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 16: Meta Selector (Train on Candidate Features)")
print("=" * 120)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Build feature matrix
X = []
y = []
meta_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)
        
        for cand in g.values():
            cand_arr = cand["solution"]
            is_gt = same_grid(cand_arr, gt)
            rank_a = find_rank(ranked_a, cand_arr)
            rank_b = find_rank(ranked_b, cand_arr)
            
            if rank_a is None:
                continue
            
            features = [
                cand["beam_score"],
                np.mean(cand["score_aug"]),
                np.std(cand["score_aug"]),
                rank_a,
                rank_b if rank_b is not None else rank_a,
            ]
            X.append(features)
            y.append(1 if is_gt else 0)
            meta_data.append({
                "bk": bk,
                "is_gt": is_gt,
                "rank_a": rank_a,
            })

X = np.array(X)
y = np.array(y)

print(f"\n📊 Dataset size: {len(X)} candidates, {sum(y)} GT candidates")

if len(X) > 5:
    # Leave-One-Out Cross-Validation (since dataset is small)
    loo = LeaveOneOut()
    predictions = []
    true_labels = []
    prob_predictions = []
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    for train_idx, test_idx in loo.split(X_scaled):
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        clf.fit(X_train, y_train)
        
        pred = clf.predict(X_test)
        prob = clf.predict_proba(X_test)[0, 1]
        
        predictions.append(pred[0])
        true_labels.append(y_test[0])
        prob_predictions.append(prob)
    
    # Evaluate
    acc = accuracy_score(true_labels, predictions)
    prec = precision_score(true_labels, predictions, zero_division=0)
    rec = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)
    
    print(f"\n📊 LOOCV Performance (Meta-Selector):")
    print(f"   Accuracy:  {acc:.4f}")
    print(f"   Precision: {prec:.4f}")
    print(f"   Recall:    {rec:.4f}")
    print(f"   F1 Score:  {f1:.4f}")
    
    # Feature Importance (Coefficients)
    # Retrain on full data to get coefficients
    clf_full = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    clf_full.fit(X_scaled, y)
    
    feature_names = ['beam', 'aug_mean', 'aug_std', 'rank_a', 'rank_b']
    print("\n📊 Feature Coefficients (Logistic Regression):")
    for name, coef in zip(feature_names, clf_full.coef_[0]):
        print(f"   {name:10s}: {coef:.4f}")
    print("   (Negative coefficient means higher value -> lower probability of being GT)")
    
    # Simulate Meta-Selector as Attempt2
    print("\n📊 Simulating Meta-Selector as Attempt 2:")
    # For each subkey, pick candidate with highest probability of being GT
    meta_score = 0.0
    num_tasks_per_puzzle = defaultdict(int)
    for bk in decoder.decoded_results.keys():
        puzzle_id = bk.rsplit("_", 1)[0]
        num_tasks_per_puzzle[puzzle_id] += 1
    
    # Group predictions by bk
    group_data = defaultdict(list)
    for i, bk in enumerate([d['bk'] for d in meta_data]):
        group_data[bk].append(i)
    
    for bk, indices in group_data.items():
        correct = decoder.dataset.replies.get(bk, [None])[0]
        if correct is None:
            continue
        
        # Get highest probability candidate for this bk
        best_idx = max(indices, key=lambda i: prob_predictions[i])
        best_candidate = None
        for cand in decoder.decoded_results[bk].values():
            if same_grid(cand["solution"], X[best_idx][0]):  # approximate, better to store cand
                best_candidate = cand["solution"]
                break
        
        # Fallback: if we can't map, just use the prediction
        if best_candidate is None:
            continue
        
        # Check if Attempt1 (Probmul Top1) or Attempt2 (Meta) solves it
        g = {k: v for k, v in decoder.decoded_results[bk].items()}
        ranked_a = score_full_probmul_3(g)
        top1 = ranked_a[0] if ranked_a else None
        
        solved = False
        if top1 is not None and same_grid(top1, correct):
            solved = True
        elif same_grid(best_candidate, correct):
            solved = True
        
        if solved:
            puzzle_id = bk.rsplit("_", 1)[0]
            meta_score += 1.0 / num_tasks_per_puzzle[puzzle_id]
    
    print(f"   Meta-Selector Score: {meta_score:.2f}")
    print(f"   Current Dual Score:  {current_score:.2f}")
    print(f"   Improvement:         {meta_score - current_score:+.2f}")
    
    # Interpret results
    print("\n📌 Interpretation:")
    if meta_score > current_score:
        print("   ✅ Meta-Selector beats current dual selection!")
        print("   → We should implement a learned selection model.")
    else:
        print("   ⚠️  Meta-Selector does NOT beat current selection.")
        print("   → The dataset is too small. Need full 172 subkeys to train effectively.")
else:
    print("⚠️  Too few candidates to train meta-selector. Run full dataset first.")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 17: 142ca369 Repair Analysis (⭐⭐⭐⭐)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 17: 142ca369 Repair Analysis")
print("=" * 120)

target_puzzle = "142ca369"

if target_puzzle in ALL_BKS:
    for bk in ALL_BKS[target_puzzle]:
        if target_puzzle not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[target_puzzle]):
            continue
        
        gt = GT[target_puzzle][sub_idx]
        gt_arr = np.asarray(gt)
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        
        # Find closest candidate
        best_sim = -1
        best_cand = None
        for cand in g.values():
            cand_arr = np.asarray(cand["solution"])
            if cand_arr.shape == gt_arr.shape:
                sim = np.mean(cand_arr == gt_arr) * 100
                if sim > best_sim:
                    best_sim = sim
                    best_cand = cand
        
        if best_cand is None:
            continue
        
        cand_arr = np.asarray(best_cand["solution"])
        diff_coords = np.argwhere(cand_arr != gt_arr)
        
        print(f"\n📌 {bk}")
        print(f"   Similarity: {best_sim:.2f}%")
        print(f"   Total differing cells: {len(diff_coords)}")
        
        if len(diff_coords) > 0:
            print("\n   🔍 Difference Pattern Analysis:")
            # Count by value
            from collections import Counter
            gt_vals = [gt_arr[r, c] for r, c in diff_coords]
            cand_vals = [cand_arr[r, c] for r, c in diff_coords]
            
            print(f"   GT values at diff cells: {Counter(gt_vals)}")
            print(f"   Candidate values at diff cells: {Counter(cand_vals)}")
            
            print("\n   📍 First 10 differing cells:")
            for i, (r, c) in enumerate(diff_coords[:10]):
                print(f"      ({r},{c}): GT={gt_arr[r,c]}, Cand={cand_arr[r,c]}")
            if len(diff_coords) > 10:
                print(f"      ... and {len(diff_coords)-10} more")
            
            # Repair possibility
            print("\n   💡 Repair Strategy:")
            print("      If similarity > 90%, a rule-based repair (e.g., majority vote on differing pixels)")
            print("      could potentially recover the exact GT.")
            print("      This is a GENERATION issue, not a SELECTION issue.")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 18: Parse Audit (⭐⭐⭐⭐)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 18: Parse Audit (Why 57% coverage?)")
print("=" * 120)

missing_analysis = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank = find_rank(ranked_a, gt)
        
        gt_present = rank is not None
        
        # Check if fallback exists
        import os
        fallback_path = f"/kaggle/inference_outputs/{bk}.fallback"
        has_fallback = os.path.exists(fallback_path)
        
        # Candidate stats
        total_cands = len(g)
        if total_cands > 0:
            unique_cands = len({hashable(c["solution"]) for c in g.values()})
            avg_sim = 0
            for cand in g.values():
                cand_arr = np.asarray(cand["solution"])
                if cand_arr.shape == np.asarray(gt).shape:
                    avg_sim += np.mean(cand_arr == np.asarray(gt)) * 100
            avg_sim = avg_sim / total_cands if total_cands > 0 else 0
        else:
            unique_cands = 0
            avg_sim = 0
        
        missing_analysis.append({
            "bk": bk,
            "gt_present": gt_present,
            "has_fallback": has_fallback,
            "total_cands": total_cands,
            "unique_cands": unique_cands,
            "avg_sim": avg_sim,
            "rank": rank,
        })

df_missing = pd.DataFrame(missing_analysis)

print("\n📊 Parse Audit Summary:")
print(df_missing.to_string(index=False))

print("\n📊 Classification of Missing GT cases:")
for _, row in df_missing.iterrows():
    if row['gt_present']:
        print(f"  ✅ {row['bk']}: GT present (rank {row['rank']})")
    else:
        if row['has_fallback']:
            print(f"  ⚠️  {row['bk']}: FALLBACK exists → Worker crashed or timed out before parsing")
        elif row['total_cands'] == 0:
            print(f"  ❌ {row['bk']}: NO candidates generated → Worker likely crashed")
        elif row['avg_sim'] > 90:
            print(f"  🔧 {row['bk']}: {row['total_cands']} candidates, avg_sim={row['avg_sim']:.1f}% → Generation is close but not exact (BEAM SEARCH issue)")
        elif row['avg_sim'] > 50:
            print(f"  🟡 {row['bk']}: {row['total_cands']} candidates, avg_sim={row['avg_sim']:.1f}% → Generation is partially correct (MODEL CAPACITY issue)")
        else:
            print(f"  🔴 {row['bk']}: {row['total_cands']} candidates, avg_sim={row['avg_sim']:.1f}% → Generation is severely off (MODEL or DATA issue)")

print("\n📌 Root Cause Summary:")
fallback_count = df_missing['has_fallback'].sum()
zero_cand = (df_missing['total_cands'] == 0).sum()
gt_present_count = df_missing['gt_present'].sum()
total_sub = len(df_missing)

print(f"   GT Present:         {gt_present_count}/{total_sub} ({gt_present_count/total_sub*100:.1f}%)")
print(f"   Fallback files:     {fallback_count}/{total_sub} (infrastructure/worker crash)")
print(f"   Zero candidates:    {zero_cand}/{total_sub} (worker did not produce output)")
print(f"   Generated but wrong: {total_sub - gt_present_count - fallback_count - zero_cand}/{total_sub} (model/beam search issue)")

print("\n" + "=" * 120)
print("✅ ALL EXPERIMENTS (15-18) COMPLETE.")
print("=" * 120)

EXPERIMENT 15: Oracle Top5 Pattern Mining

📌 1ae2feb7_2 (GT Rank: 1)
--------------------------------------------------------------------------------
Rank   Beam         Aug Mean     Aug Std      Is GT   
--------------------------------------------------------------------------------
1      0.0105       2.9581       2.6728       True     <== GT
2      0.5211       7.2979       4.2022       False   
3      0.2181       8.7128       3.5383       False   
4      1.2702       11.8322      6.3046       False   
5      1.3757       12.7791      9.1302       False   

📌 d8e07eb2_1 (GT Rank: 1)
--------------------------------------------------------------------------------
Rank   Beam         Aug Mean     Aug Std      Is GT   
--------------------------------------------------------------------------------
1      1.1451       1.4008       1.6827       True     <== GT
2      0.9165       3.8042       3.0521       False   
3      0.5235       10.1892      4.1022       False   
4      0.0859   

In [31]:
# ================================================================
# SETUP for Experiments 19-21
# ================================================================
# ================================================================
# EXPERIMENT 19: Beam Search Autopsy (Where does GT branch go?)
# ================================================================

# 🔧 FIX: Define constants (moved from arc_solver.py)
MAX_BEAM_CANDIDATES = 10
MAX_SCORE = -np.log(0.2)  # ≈ 1.609

print("=" * 120)
print("EXPERIMENT 19: Beam Search Autopsy (Where does GT branch go?)")
print("=" * 120)

autopsy_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        rank_gt = find_rank(ranked_a, gt)
        
        closest_sim = -1.0
        closest_beam = None
        closest_rank = None
        gt_present = rank_gt is not None
        
        if not gt_present:
            for cand in g.values():
                cand_arr = np.asarray(cand["solution"])
                if cand_arr.shape == np.asarray(gt).shape:
                    sim = np.mean(cand_arr == np.asarray(gt)) * 100
                    if sim > closest_sim:
                        closest_sim = sim
                        closest_beam = cand["beam_score"]
                        closest_rank = find_rank(ranked_a, cand["solution"])
        
        if gt_present:
            gt_beam = None
            for cand in g.values():
                if same_grid(cand["solution"], gt):
                    gt_beam = cand["beam_score"]
                    break
            is_pruned = gt_beam is not None and gt_beam > MAX_SCORE
            status = f"PRESENT (Rank {rank_gt}, beam={gt_beam:.4f})"
            if is_pruned:
                status += " ⚠️ Would be pruned by max_score!"
            else:
                status += " ✅ Survives beam pruning"
        else:
            if closest_beam is None:
                status = f"MISSING (No candidate with matching shape)"
            else:
                status = f"MISSING (Closest sim={closest_sim:.2f}%, beam={closest_beam:.4f}, rank={closest_rank})"
            
            if closest_beam is not None and closest_beam > MAX_SCORE:
                status += " ⚠️ Pruned by max_score!"
            elif closest_rank is not None and closest_rank > MAX_BEAM_CANDIDATES:
                status += f" ⚠️ Exceeds beam width ({MAX_BEAM_CANDIDATES})!"
            else:
                status += " ❓ Unknown cause"
        
        autopsy_data.append({
            "bk": bk,
            "gt_present": gt_present,
            "rank_gt": rank_gt,
            "status": status,
        })
        print(f"  {bk}: {status}")

print("\n📊 Beam Search Autopsy Summary:")
df_autopsy = pd.DataFrame(autopsy_data)
present = df_autopsy['gt_present'].sum()
total = len(df_autopsy)
print(f"  GT Present: {present}/{total} ({present/total*100:.1f}%)")
print(f"  GT Missing: {total-present}/{total} ({(total-present)/total*100:.1f}%)")

missing_df = df_autopsy[df_autopsy['gt_present'] == False]
if len(missing_df) > 0:
    print("\n🔍 Missing GT Cases Analysis:")
    for _, row in missing_df.iterrows():
        print(f"  {row['bk']}: {row['status']}")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 20: Search Hyperparameter Sweep (Simulated)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 20: Search Hyperparameter Sweep (Simulated)")
print("=" * 120)



for beam_width in [1, 2, 3, 5, 10, 20]:
    covered = 0
    for puzzle_id, subkeys in ALL_BKS.items():
        for bk in subkeys:
            if puzzle_id not in GT:
                continue
            sub_idx = int(bk.rsplit("_", 1)[1])
            if sub_idx >= len(GT[puzzle_id]):
                continue
            
            gt = GT[puzzle_id][sub_idx]
            guesses = decoder.decoded_results[bk]
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            rank_gt = find_rank(ranked_a, gt)
            
            if rank_gt is not None and rank_gt <= beam_width:
                covered += 1
    
    pct = covered / total * 100 if total > 0 else 0
    print(f"  Beam Width {beam_width:2d}: {covered:2d}/{total} ({pct:.1f}%) puzzles have GT in Top-{beam_width}")

print("\n📊 Effect of lowering max_score (less pruning):")
# Current max_score = -log(0.2) = 1.609
# Simulate what happens if we allow higher scores (less pruning)
for max_score_factor in [0.5, 0.8, 1.0, 1.5, 2.0]:
    threshold = MAX_SCORE * max_score_factor
    # Check if GT beam scores are below threshold
    within_threshold = 0
    for puzzle_id, subkeys in ALL_BKS.items():
        for bk in subkeys:
            if puzzle_id not in GT:
                continue
            sub_idx = int(bk.rsplit("_", 1)[1])
            if sub_idx >= len(GT[puzzle_id]):
                continue
            
            gt = GT[puzzle_id][sub_idx]
            guesses = decoder.decoded_results[bk]
            g = {k: v for k, v in guesses.items()}
            
            gt_beam = None
            for cand in g.values():
                if same_grid(cand["solution"], gt):
                    gt_beam = cand["beam_score"]
                    break
            
            if gt_beam is not None and gt_beam <= threshold:
                within_threshold += 1
    
    print(f"  max_score = {threshold:.3f}: {within_threshold}/{total} GT candidates survive pruning")



print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 21: Candidate Diversity Audit
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 21: Candidate Diversity Audit")
print("=" * 120)

diversity_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        gt = GT[puzzle_id][sub_idx]
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        total_cands = len(g)
        unique_hashes = set()
        hash_counts = Counter()
        for cand in g.values():
            h = hashable(cand["solution"])
            unique_hashes.add(h)
            hash_counts[h] += 1
        
        unique_cands = len(unique_hashes)
        dup_ratio = 1 - (unique_cands / total_cands) if total_cands > 0 else 0
        
        # Most common duplicate
        most_common = hash_counts.most_common(1)
        most_common_count = most_common[0][1] if most_common else 0
        
        diversity_data.append({
            "bk": bk,
            "total": total_cands,
            "unique": unique_cands,
            "dup_ratio": dup_ratio,
            "most_common_dup": most_common_count,
            "gt_present": find_rank(score_full_probmul_3(g), gt) is not None,
        })

df_diversity = pd.DataFrame(diversity_data)
print("\n📊 Diversity Summary:")
print(df_diversity.to_string(index=False))

print("\n📊 Aggregate Statistics:")
print(f"  Average candidates per puzzle: {df_diversity['total'].mean():.1f}")
print(f"  Average unique candidates:     {df_diversity['unique'].mean():.1f}")
print(f"  Average duplicate ratio:       {df_diversity['dup_ratio'].mean():.3f}")
print(f"  Most duplicated candidate count: {df_diversity['most_common_dup'].max()}")

print("\n📌 Diversity Analysis:")
low_diversity = df_diversity[df_diversity['dup_ratio'] > 0.5]
if len(low_diversity) > 0:
    print(f"  Puzzles with >50% duplicates: {low_diversity['bk'].tolist()}")
    print("  → Beam search is not exploring enough diverse branches.")


print("\n" + "=" * 120)


# ================================================================
# BONUS: 142ca369 Beam Trace Visualization (FIXED)
# ================================================================

print("\n" + "=" * 120)
print("BONUS: 142ca369 Beam Trace Visualization")
print("=" * 120)

target_puzzle = "142ca369"

if target_puzzle in ALL_BKS:
    for bk in ALL_BKS[target_puzzle]:
        if target_puzzle not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[target_puzzle]):
            continue
        
        gt = GT[target_puzzle][sub_idx]
        gt_arr = np.asarray(gt)
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        ranked_a = score_full_probmul_3(g)
        
        print(f"\n📌 Beam Trace for {bk}")
        print("-" * 80)
        print(f"{'Rank':<6} {'Beam Score':<14} {'Aug Mean':<12} {'Similarity to GT':<16} {'Candidate Hash (first 20)'}")
        print("-" * 80)
        
        for i, cand_grid in enumerate(ranked_a[:15], start=1):
            # cand_grid is a numpy array (the grid itself)
            cand_arr = np.asarray(cand_grid)
            sim = 0
            if cand_arr.shape == gt_arr.shape:
                sim = np.mean(cand_arr == gt_arr) * 100
            
            # Find features for this candidate grid
            features = None
            for c in g.values():
                if same_grid(c["solution"], cand_grid):
                    features = c
                    break
            
            if features is None:
                # If features not found, skip or print placeholders
                aug_mean = 0.0
                beam_score = 0.0
            else:
                aug_mean = np.mean(features["score_aug"])
                beam_score = features["beam_score"]
            
            h = hashable(cand_grid)
            h_str = str(h)[:20] + "..." if len(str(h)) > 20 else str(h)
            
            mark = " <== GT" if same_grid(cand_grid, gt) else ""
            print(f"{i:<6} {beam_score:<14.4f} {aug_mean:<12.4f} {sim:<16.2f}% {h_str}{mark}")
        
       

print("\n" + "=" * 120)
print("✅ EXPERIMENTS 19-21 COMPLETE.")
print("=" * 120)

EXPERIMENT 19: Beam Search Autopsy (Where does GT branch go?)
  b99e7126_0: MISSING (Closest sim=96.67%, beam=0.8256, rank=2) ❓ Unknown cause
  4c7dc4dd_1: MISSING (Closest sim=66.67%, beam=0.5976, rank=13) ⚠️ Exceeds beam width (10)!
  4c7dc4dd_0: MISSING (Closest sim=20.00%, beam=1.3230, rank=6) ❓ Unknown cause
  88bcf3b4_0: MISSING (Closest sim=95.87%, beam=1.3573, rank=1) ❓ Unknown cause
  88bcf3b4_1: MISSING (Closest sim=96.02%, beam=0.7939, rank=5) ❓ Unknown cause
  1ae2feb7_0: MISSING (Closest sim=98.00%, beam=0.5066, rank=3) ❓ Unknown cause
  1ae2feb7_1: MISSING (Closest sim=99.17%, beam=0.0248, rank=1) ❓ Unknown cause
  1ae2feb7_2: PRESENT (Rank 1, beam=0.0105) ✅ Survives beam pruning
  d8e07eb2_1: PRESENT (Rank 1, beam=1.1451) ✅ Survives beam pruning
  d8e07eb2_0: MISSING (Closest sim=98.64%, beam=1.2991, rank=2) ❓ Unknown cause
  78332cb0_1: PRESENT (Rank 1, beam=0.4337) ✅ Survives beam pruning
  78332cb0_0: MISSING (Closest sim=97.39%, beam=1.5835, rank=8) ❓ Unknown cause
 

In [32]:
# ================================================================
# EXPERIMENT 22: Duplicate Source Analysis
# ================================================================

print("=" * 120)
print("EXPERIMENT 22: Duplicate Source Analysis")
print("=" * 120)

dup_analysis = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        guesses = decoder.decoded_results[bk]
        g = {k: v for k, v in guesses.items()}
        
        # Group by hash
        hash_groups = {}
        for key, cand in g.items():
            h = hashable(cand["solution"])
            if h not in hash_groups:
                hash_groups[h] = []
            hash_groups[h].append(key)
        
        # Analyze duplicates
        total = len(g)
        unique = len(hash_groups)
        dup_count = total - unique
        
        # Find most duplicated
        most_dup = max([len(v) for v in hash_groups.values()]) if hash_groups else 0
        
        # Check if duplicates share similar beam scores (same branch?)
        dup_pairs = []
        for h, keys in hash_groups.items():
            if len(keys) > 1:
                beam_scores = [g[k]["beam_score"] for k in keys]
                dup_pairs.append({
                    "hash": h,
                    "count": len(keys),
                    "beam_min": min(beam_scores),
                    "beam_max": max(beam_scores),
                    "beam_range": max(beam_scores) - min(beam_scores),
                })
        
        dup_analysis.append({
            "bk": bk,
            "total": total,
            "unique": unique,
            "dup_count": dup_count,
            "dup_ratio": dup_count / total if total > 0 else 0,
            "most_duplicated": most_dup,
            "num_unique_hashes": len(hash_groups),
            "dup_pairs": dup_pairs,
        })

# Display summary
print("\n📊 Duplicate Source Summary:")
for item in dup_analysis:
    print(f"\n  {item['bk']}:")
    print(f"    Total: {item['total']}, Unique: {item['unique']}, Duplicates: {item['dup_count']} ({item['dup_ratio']*100:.1f}%)")
    print(f"    Most duplicated: {item['most_duplicated']} copies")
    if item['dup_pairs']:
        print("    Duplicate groups (hash, count, beam_range):")
        for p in item['dup_pairs'][:3]:
            print(f"      {str(p['hash'])[:20]}... x{p['count']}, beam range: {p['beam_range']:.4f}")



print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 23: Beam Path Diversity (Grid-level)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 23: Beam Path Diversity (Pairwise Similarity)")
print("=" * 120)

diversity_results = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[puzzle_id]):
            continue
        
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        if len(g) < 2:
            continue
        
        # Compute pairwise similarities
        sim_matrix = []
        for i in range(min(len(g), 15)):
            row = []
            for j in range(min(len(g), 15)):
                if i == j:
                    row.append(100.0)
                else:
                    a = np.asarray(g[i]["solution"])
                    b = np.asarray(g[j]["solution"])
                    if a.shape == b.shape:
                        sim = np.mean(a == b) * 100
                    else:
                        sim = 0.0
                    row.append(sim)
            sim_matrix.append(row)
        
        # Average similarity (excluding self)
        avg_sim = 0
        count = 0
        for i in range(len(sim_matrix)):
            for j in range(len(sim_matrix)):
                if i != j:
                    avg_sim += sim_matrix[i][j]
                    count += 1
        avg_sim = avg_sim / count if count > 0 else 0
        
        # High similarity pairs (>90%)
        high_sim_pairs = 0
        for i in range(len(sim_matrix)):
            for j in range(i+1, len(sim_matrix)):
                if sim_matrix[i][j] > 90:
                    high_sim_pairs += 1
        
        diversity_results.append({
            "bk": bk,
            "num_candidates": len(sim_matrix),
            "avg_pairwise_sim": avg_sim,
            "high_sim_pairs": high_sim_pairs,
            "total_pairs": len(sim_matrix) * (len(sim_matrix) - 1) // 2,
        })

print("\n📊 Beam Diversity Metrics:")
for r in diversity_results:
    pct = r['high_sim_pairs'] / r['total_pairs'] * 100 if r['total_pairs'] > 0 else 0
    print(f"  {r['bk']}:")
    print(f"    Avg pairwise similarity: {r['avg_pairwise_sim']:.2f}%")
    print(f"    High-sim pairs (>90%): {r['high_sim_pairs']}/{r['total_pairs']} ({pct:.1f}%)")



print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 24: Cache Reuse Audit (Code Review)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 24: Cache Reuse Audit (turbo_dfs code review)")
print("=" * 120)

# Read arc_solver.py to inspect turbo_dfs
import os
import re

solver_path = "/kaggle/working/arc_solver.py"
if os.path.exists(solver_path):
    with open(solver_path, "r") as f:
        solver_code = f.read()
    
    # Find turbo_dfs function
    start = solver_code.find("def turbo_dfs")
    if start != -1:
        # Extract the function body
        end = solver_code.find("\n\n", start + 100)
        if end == -1:
            end = len(solver_code)
        func_code = solver_code[start:end]
        
        print("\n📄 turbo_dfs function (excerpt):")
        print("-" * 80)
        print(func_code[:1500] + ("..." if len(func_code) > 1500 else ""))
        print("-" * 80)
        
        # Check for cache-related issues
        print("\n🔍 Cache Reuse Audit:")
        issues = []
        
        # Check if past_key_values is reused without cloning
        if "past_key_values=cache" in func_code:
            issues.append("⚠️  Cache is passed directly without cloning → may cause mutation issues")
        
        if "next_cache = outputs.past_key_values" in func_code:
            issues.append("⚠️  Cache is mutated in-place during recursive calls")
        
        if "del outputs" in func_code:
            issues.append("ℹ️  outputs deleted but cache remains → could cause reference issues")
        
        # Check for diversity penalty
        if "diversity" not in func_code.lower() and "penalty" not in func_code.lower():
            issues.append("⚠️  No diversity penalty in beam search → beams may collapse")
        
        if issues:
            for issue in issues:
                print(f"  {issue}")
        else:
            print("  ✅ No obvious cache issues found.")
        
        

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 25: Token Frequency Analysis (142ca369)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 25: Token Frequency Analysis (142ca369)")
print("=" * 120)

target = "142ca369"
if target in ALL_BKS:
    for bk in ALL_BKS[target]:
        if target not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[target]):
            continue
        
        gt = GT[target][sub_idx]
        gt_arr = np.asarray(gt)
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        print(f"\n📌 {bk}: Token Frequency Analysis")
        print("-" * 80)
        
        # Collect all candidate grids
        all_grids = []
        for cand in g:
            arr = np.asarray(cand["solution"])
            if arr.shape == gt_arr.shape:
                all_grids.append(arr)
        
        if not all_grids:
            continue
        
        # Per-position token frequency
        token_freq = {}
        for pos in np.ndindex(gt_arr.shape):
            values = [grid[pos] for grid in all_grids]
            counter = Counter(values)
            token_freq[pos] = counter
        
        # Find positions where GT token is not the most frequent
        print("  Positions where GT token is NOT the majority:")
        mismatch_count = 0
        for pos, counter in token_freq.items():
            gt_val = gt_arr[pos]
            if gt_val not in counter:
                print(f"    {pos}: GT={gt_val} → NOT PRESENT in any candidate")
                mismatch_count += 1
            elif counter[gt_val] < len(all_grids) * 0.5:
                most_common = counter.most_common(1)[0]
                print(f"    {pos}: GT={gt_val} → majority is {most_common[0]} ({most_common[1]}/{len(all_grids)})")
                mismatch_count += 1
        
        if mismatch_count == 0:
            print("    ✅ GT token is majority or present in all positions.")
        
        # Token transition patterns (adjacent differences)
        print("\n  Token transition patterns (GT → Candidate mapping):")
        # For closest candidate
        closest_sim = -1
        closest_cand = None
        for cand in g:
            arr = np.asarray(cand["solution"])
            if arr.shape == gt_arr.shape:
                sim = np.mean(arr == gt_arr) * 100
                if sim > closest_sim:
                    closest_sim = sim
                    closest_cand = arr
        
        if closest_cand is not None:
            diff_coords = np.argwhere(closest_cand != gt_arr)
            if len(diff_coords) > 0:
                transitions = Counter()
                for r, c in diff_coords:
                    gt_val = gt_arr[r, c]
                    cand_val = closest_cand[r, c]
                    transitions[(gt_val, cand_val)] += 1
                
                print("    GT→Candidate transitions:")
                for (gt_val, cand_val), count in transitions.most_common(10):
                    print(f"      {gt_val} → {cand_val}: {count} times")

print("\n" + "=" * 120)
print("✅ EXPERIMENTS 22-25 COMPLETE.")
print("=" * 120)

EXPERIMENT 22: Duplicate Source Analysis

📊 Duplicate Source Summary:

  b99e7126_0:
    Total: 18, Unique: 6, Duplicates: 12 (66.7%)
    Most duplicated: 11 copies
    Duplicate groups (hash, count, beam_range):
      ((np.int64(8), np.in... x11, beam range: 0.0583
      ((np.int64(8), np.in... x3, beam range: 0.6184

  4c7dc4dd_1:
    Total: 17, Unique: 14, Duplicates: 3 (17.6%)
    Most duplicated: 3 copies
    Duplicate groups (hash, count, beam_range):
      ((np.int64(0), np.in... x3, beam range: 1.2293
      ((np.int64(0), np.in... x2, beam range: 0.8075

  4c7dc4dd_0:
    Total: 22, Unique: 7, Duplicates: 15 (68.2%)
    Most duplicated: 11 copies
    Duplicate groups (hash, count, beam_range):
      ((np.int64(0), np.in... x11, beam range: 1.5663
      ((np.int64(0), np.in... x2, beam range: 0.7963
      ((np.int64(8), np.in... x3, beam range: 0.5103

  88bcf3b4_0:
    Total: 16, Unique: 15, Duplicates: 1 (6.2%)
    Most duplicated: 2 copies
    Duplicate groups (hash, count, b

In [33]:
# ================================================================
# EXPERIMENT 26: Duplicate Origin Audit (Pipeline Trace)
# ================================================================

print("=" * 120)
print("EXPERIMENT 26: Duplicate Origin Audit (Pipeline Trace)")
print("=" * 120)

# Import required modules
import bz2
import pickle
import os
from collections import defaultdict

def trace_pipeline(bk):
    """
    Trace a single subkey through the entire pipeline:
    decoded_results → ranked → dedup → final
    """
    if bk not in decoder.decoded_results:
        return None
    
    guesses = decoder.decoded_results[bk]
    g = {k: v for k, v in guesses.items()}
    
    # Stage 1: Raw candidates (from decoded_results)
    stage1_total = len(g)
    stage1_hashes = [hashable(c["solution"]) for c in g.values()]
    stage1_unique = len(set(stage1_hashes))
    
    # Stage 2: After ranking (Probmul)
    ranked_a = score_full_probmul_3(g)
    stage2_total = len(ranked_a)
    stage2_hashes = [hashable(c) for c in ranked_a]
    stage2_unique = len(set(stage2_hashes))
    
    # Stage 3: After dedup (unique candidates only)
    unique_cands = {}
    for cand in g.values():
        h = hashable(cand["solution"])
        if h not in unique_cands:
            unique_cands[h] = cand
    stage3_total = len(unique_cands)
    
    # Stage 4: After selection (current dual)
    current_dual = decoder.run_dual_selection().get(bk, [])
    stage4_total = len(current_dual)
    
    return {
        "bk": bk,
        "stage1_total": stage1_total,
        "stage1_unique": stage1_unique,
        "stage2_total": stage2_total,
        "stage2_unique": stage2_unique,
        "stage3_total": stage3_total,
        "stage4_total": stage4_total,
        "dup_ratio_stage1": 1 - (stage1_unique / stage1_total) if stage1_total > 0 else 0,
        "dup_ratio_stage2": 1 - (stage2_unique / stage2_total) if stage2_total > 0 else 0,
    }

print("\n📊 Pipeline Trace for all subkeys:")
trace_data = []
for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        trace = trace_pipeline(bk)
        if trace:
            trace_data.append(trace)

# Display as table
print(f"\n{'Subkey':<15} {'Raw':<6} {'Unique':<7} {'Dup%':<7} {'Ranked':<7} {'RankUniq':<9} {'AfterDedup':<11} {'Selected':<10}")
print("-" * 90)
for t in trace_data:
    print(f"{t['bk']:<15} {t['stage1_total']:<6} {t['stage1_unique']:<7} {t['dup_ratio_stage1']*100:<7.1f}% {t['stage2_total']:<7} {t['stage2_unique']:<9} {t['stage3_total']:<11} {t['stage4_total']:<10}")

print("\n📊 Aggregate Statistics:")
if trace_data:
    avg_dup1 = np.mean([t['dup_ratio_stage1'] for t in trace_data])
    avg_dup2 = np.mean([t['dup_ratio_stage2'] for t in trace_data])
    print(f"  Average duplicate ratio (raw):    {avg_dup1*100:.1f}%")
    print(f"  Average duplicate ratio (ranked): {avg_dup2*100:.1f}%")
    print(f"  Total unique candidates:          {sum(t['stage3_total'] for t in trace_data)}")
    print(f"  Total selected by dual:           {sum(t['stage4_total'] for t in trace_data)}")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 27: Beam Collapse Audit (Parent-Child Analysis)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 27: Beam Collapse Audit (Parent-Child Analysis)")
print("=" * 120)

print("\n📊 Simulated Beam Collapse Analysis:")
print("(Using candidate similarity as proxy for parent-child relationship)")

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        if len(g) < 5:
            continue
        
        # Compute pairwise similarities for first 10 candidates
        sim_matrix = []
        for i in range(min(10, len(g))):
            row = []
            for j in range(min(10, len(g))):
                if i == j:
                    row.append(100.0)
                else:
                    a = np.asarray(g[i]["solution"])
                    b = np.asarray(g[j]["solution"])
                    if a.shape == b.shape:
                        sim = np.mean(a == b) * 100
                    else:
                        sim = 0.0
                    row.append(sim)
            sim_matrix.append(row)
        
        # Count high-sim pairs (>95%)
        high_sim = 0
        total_pairs = 0
        for i in range(len(sim_matrix)):
            for j in range(i+1, len(sim_matrix)):
                total_pairs += 1
                if sim_matrix[i][j] > 95:
                    high_sim += 1
        
        collapse_ratio = high_sim / total_pairs if total_pairs > 0 else 0
        
        print(f"\n  {bk}:")
        print(f"    Candidates analyzed: {len(sim_matrix)}")
        print(f"    High-sim pairs (>95%): {high_sim}/{total_pairs} ({collapse_ratio*100:.1f}%)")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 28: convert_tokens_to_array Audit
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 28: convert_tokens_to_array Audit")
print("=" * 120)

target_bk = "142ca369_1"

if target_bk in decoder.decoded_results:
    guesses = decoder.decoded_results[target_bk]
    g = {k: v for k, v in guesses.items()}
    
    gt = GT["142ca369"][1]
    gt_arr = np.asarray(gt)
    
    closest = None
    closest_sim = -1
    for cand in g.values():
        cand_arr = np.asarray(cand["solution"])
        if cand_arr.shape == gt_arr.shape:
            sim = np.mean(cand_arr == gt_arr) * 100
            if sim > closest_sim:
                closest_sim = sim
                closest = cand
    
    if closest is not None:
        print(f"\n📌 Analyzing token→grid conversion for {target_bk}")
        print(f"   Closest candidate similarity: {closest_sim:.2f}%")
        
        cand_arr = np.asarray(closest["solution"])
        print(f"   Candidate grid shape: {cand_arr.shape}")
        print(f"   GT grid shape:        {gt_arr.shape}")
        
        if cand_arr.shape == gt_arr.shape:
            diff_coords = np.argwhere(cand_arr != gt_arr)
            print(f"   Differing cells: {len(diff_coords)}")
            
            if len(diff_coords) > 0:
                print("\n   🔍 Pattern Analysis of Differing Cells:")
                rows = defaultdict(list)
                for r, c in diff_coords:
                    rows[int(r)].append(int(c))
                
                print("   Differences by row:")
                for r in sorted(rows.keys()):
                    print(f"     Row {r}: columns {rows[r][:10]} (total: {len(rows[r])})")
        else:
            print(f"   ⚠️  Shape mismatch! Candidate: {cand_arr.shape}, GT: {gt_arr.shape}")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 29: Token Transition Audit (Systematic Pattern)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 29: Token Transition Audit (Systematic Pattern)")
print("=" * 120)

target = "142ca369"

if target in ALL_BKS:
    for bk in ALL_BKS[target]:
        if target not in GT:
            continue
        sub_idx = int(bk.rsplit("_", 1)[1])
        if sub_idx >= len(GT[target]):
            continue
        
        gt = GT[target][sub_idx]
        gt_arr = np.asarray(gt)
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        print(f"\n📌 {bk}: Token Transition Analysis")
        print("-" * 80)
        
        closest_sim = -1
        closest_cand = None
        for cand in g:
            arr = np.asarray(cand["solution"])
            if arr.shape == gt_arr.shape:
                sim = np.mean(arr == gt_arr) * 100
                if sim > closest_sim:
                    closest_sim = sim
                    closest_cand = arr
        
        if closest_cand is None:
            continue
        
        diff_coords = np.argwhere(closest_cand != gt_arr)
        
        if len(diff_coords) == 0:
            print("  ✅ Exact match found!")
            continue
        
        transitions = defaultdict(int)
        for r, c in diff_coords:
            gt_val = int(gt_arr[int(r), int(c)])
            cand_val = int(closest_cand[int(r), int(c)])
            transitions[(gt_val, cand_val)] += 1
        
        print("\n  🔍 Token Transition Patterns:")
        for (gt_val, cand_val), count in sorted(transitions.items(), key=lambda x: -x[1]):
            pct = count / len(diff_coords) * 100
            print(f"    {gt_val} → {cand_val}: {count} times ({pct:.1f}%)")
        
        # Check for systematic pattern
        print("\n  📊 Systematic Pattern Detection:")
        shifts = []
        for (gt_val, cand_val), count in transitions.items():
            if count > 2:
                shifts.append((gt_val, cand_val, count))
        
        if shifts:
            print("  ⚠️  Systematic shifts detected:")
            for gt_val, cand_val, count in shifts:
                print(f"    {gt_val} → {cand_val}: {count} times")
            
            # Check if values are shifted by a constant (FIXED: unpack correctly)
            unique_gt = sorted(set(gt_val for gt_val, _, _ in shifts))
            unique_cand = sorted(set(cand_val for _, cand_val, _ in shifts))
            if len(unique_gt) == len(unique_cand) == 3:
                print("\n  🔬 Hypothesis: Color shift pattern")
                for i in range(len(unique_gt)):
                    print(f"    {unique_gt[i]} → {unique_cand[i]} (shift: {unique_cand[i] - unique_gt[i]})")

print("\n" + "=" * 120)
print("✅ EXPERIMENTS 26-29 COMPLETE.")
print("=" * 120)

EXPERIMENT 26: Duplicate Origin Audit (Pipeline Trace)

📊 Pipeline Trace for all subkeys:

Subkey          Raw    Unique  Dup%    Ranked  RankUniq  AfterDedup  Selected  
------------------------------------------------------------------------------------------
b99e7126_0      18     6       66.7   % 6       6         6           2         
4c7dc4dd_1      17     14      17.6   % 14      14        14          2         
4c7dc4dd_0      22     7       68.2   % 7       7         7           2         
88bcf3b4_0      16     15      6.2    % 15      15        15          2         
88bcf3b4_1      7      5       28.6   % 5       5         5           2         
1ae2feb7_0      12     9       25.0   % 9       9         9           2         
1ae2feb7_1      17     3       82.4   % 3       3         3           2         
1ae2feb7_2      23     14      39.1   % 14      14        14          2         
d8e07eb2_1      26     10      61.5   % 10      10        10          2         
d8e07eb2_

In [34]:
# ================================================================
# EXPERIMENT 30: turbo_dfs Line-by-Line Audit
# ================================================================

import os
import re
import ast
import inspect
from textwrap import dedent

print("=" * 120)
print("EXPERIMENT 30: turbo_dfs Line-by-Line Audit")
print("=" * 120)

# Read arc_solver.py
solver_path = "/kaggle/working/arc_solver.py"
if os.path.exists(solver_path):
    with open(solver_path, "r") as f:
        solver_code = f.read()
    
    # Extract turbo_dfs function
    start = solver_code.find("def turbo_dfs")
    if start != -1:
        # Find the function body
        end = solver_code.find("\n\ndef ", start + 10)
        if end == -1:
            end = len(solver_code)
        func_code = solver_code[start:end]
        
        print("\n📄 COMPLETE turbo_dfs FUNCTION:")
        print("=" * 80)
        print(func_code)
        print("=" * 80)
        
        # Line-by-line analysis
        print("\n🔍 LINE-BY-LINE ANALYSIS:")
        lines = func_code.split('\n')
        
        issues = []
        for i, line in enumerate(lines, 1):
            line_stripped = line.strip()
            if not line_stripped:
                continue
            
            # Check for cache mutation
            if 'cache' in line_stripped and '=' in line_stripped:
                if 'past_key_values' in line_stripped or 'cache =' in line_stripped:
                    if '.clone()' not in line_stripped and 'copy' not in line_stripped:
                        issues.append({
                            'line': i,
                            'code': line_stripped,
                            'issue': 'Cache assignment without clone/copy - potential mutation'
                        })
            
            # Check for diversity penalty
            if 'candidates' in line_stripped and 'append' in line_stripped:
                if 'diversity' not in line_stripped.lower() and 'penalty' not in line_stripped.lower():
                    issues.append({
                        'line': i,
                        'code': line_stripped,
                        'issue': 'Candidate selection without diversity penalty'
                    })
            
            # Check for beam pruning
            if 'max_score' in line_stripped and ('if' in line_stripped or '>' in line_stripped or '<' in line_stripped):
                issues.append({
                    'line': i,
                    'code': line_stripped,
                    'issue': 'Beam pruning threshold applied'
                })
            
            # Check for position_ids
            if 'position_ids' in line_stripped:
                if 'full' not in line_stripped and 'clone' not in line_stripped:
                    issues.append({
                        'line': i,
                        'code': line_stripped,
                        'issue': 'position_ids may be reused without proper handling'
                    })
            
            # Check for recursion
            if 'turbo_dfs(' in line_stripped:
                issues.append({
                    'line': i,
                    'code': line_stripped,
                    'issue': 'Recursive call - check for infinite recursion or stack overflow'
                })
        
        print("\n🚨 POTENTIAL ISSUES FOUND:")
        for issue in issues:
            print(f"  Line {issue['line']:3d}: {issue['code'][:60]}...")
            print(f"         → {issue['issue']}")
        
        print("\n📌 CRITICAL OBSERVATIONS:")
        # Count key operations
        cache_ops = sum(1 for line in lines if 'past_key_values' in line)
        candidate_ops = sum(1 for line in lines if 'candidates' in line)
        score_ops = sum(1 for line in lines if 'score' in line)
        
        print(f"  Cache operations:   {cache_ops}")
        print(f"  Candidate ops:      {candidate_ops}")
        print(f"  Score calculations: {score_ops}")
        
        # Check if cache is ever cloned
        if 'clone' not in func_code and 'copy' not in func_code:
            print("  ⚠️  Cache is NEVER cloned or copied -> high risk of mutation")
        
        # Check for diversity
        if 'diversity' not in func_code.lower():
            print("  ⚠️  NO diversity penalty in beam search -> beams will collapse")
        
        # Check max beam candidates
        if 'MAX_BEAM_CANDIDATES' in func_code:
            max_beam = re.search(r'MAX_BEAM_CANDIDATES\s*=\s*(\d+)', func_code)
            if max_beam:
                print(f"  MAX_BEAM_CANDIDATES = {max_beam.group(1)}")
        
        # Check max_score
        if 'max_score' in func_code:
            max_score_vals = re.findall(r'max_score\s*=\s*([\d.]+)', func_code)
            if max_score_vals:
                print(f"  max_score = {max_score_vals[0]}")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 31 (FIXED): convert_tokens_to_array Audit
# ================================================================

import re

print("=" * 120)
print("EXPERIMENT 31: convert_tokens_to_array Audit (arc_loader.py)")
print("=" * 120)

# Read arc_loader.py
loader_path = "/kaggle/working/arc_loader.py"
if os.path.exists(loader_path):
    with open(loader_path, "r") as f:
        loader_code = f.read()
    
    # Extract convert_tokens_to_array function
    start = loader_code.find("def convert_tokens_to_array")
    if start != -1:
        end = loader_code.find("\n\n", start + 50)
        if end == -1:
            end = len(loader_code)
        func_code = loader_code[start:end]
        
        print("\n📄 COMPLETE convert_tokens_to_array FUNCTION:")
        print("=" * 80)
        print(func_code)
        print("=" * 80)
        
        # Line-by-line analysis
        print("\n🔍 LINE-BY-LINE ANALYSIS:")
        lines = func_code.split('\n')
        
        issues = []
        for i, line in enumerate(lines, 1):
            line_stripped = line.strip()
            if not line_stripped:
                continue
            
            # Check for EOS handling (token 15)
            if '15' in line_stripped or 'EOS' in line_stripped:
                if 'if' not in line_stripped and 'skip' not in line_stripped:
                    issues.append({
                        'line': i,
                        'code': line_stripped,
                        'issue': 'EOS token may not be properly handled'
                    })
            
            # Check for padding handling
            if 'PAD' in line_stripped or '13' in line_stripped:
                if 'if' not in line_stripped and 'skip' not in line_stripped:
                    issues.append({
                        'line': i,
                        'code': line_stripped,
                        'issue': 'Padding token may not be properly handled'
                    })
            
            # Check for newline handling
            if '\\n' in line_stripped:
                # Good, newline handling present
                pass
            
            # Check for row stripping
            if 'strip' in line_stripped:
                issues.append({
                    'line': i,
                    'code': line_stripped,
                    'issue': 'Row stripping present - good'
                })
            
            # Check for grid validation
            if 'is_valid_solution' in line_stripped:
                issues.append({
                    'line': i,
                    'code': line_stripped,
                    'issue': 'Grid validation present'
                })
        
        print("\n🚨 POTENTIAL ISSUES FOUND:")
        for issue in issues:
            print(f"  Line {issue['line']:3d}: {issue['code'][:60]}...")
            print(f"         → {issue['issue']}")
        
        # Check for specific token values
        print("\n📊 TOKEN MAPPING ANALYSIS:")
        if 'ARC_VOCAB' in loader_code:
            vocab_start = loader_code.find("ARC_VOCAB = {")
            if vocab_start != -1:
                vocab_end = loader_code.find("}", vocab_start)
                vocab = loader_code[vocab_start:vocab_end+1]
                print(f"  VOCAB mapping found:")
                print(f"    {vocab[:200]}...")
        
        # Check if tokens are properly converted to digits
        print("\n🔢 DIGIT CONVERSION ANALYSIS:")
        if 'isdigit' in func_code:
            print("  ✅ isdigit() used to filter digits")
        else:
            print("  ⚠️  No isdigit() check - may include non-digit tokens")
        
        if 'int(x)' in func_code or 'int(x' in func_code:
            print("  ✅ int() conversion present")
        else:
            print("  ⚠️  No int() conversion - may fail on non-digit strings")
        
        print("\n📌 CRITICAL OBSERVATIONS:")
        # Count key operations
        row_ops = sum(1 for line in lines if 'row' in line.lower())
        grid_ops = sum(1 for line in lines if 'grid' in line.lower())
        token_ops = sum(1 for line in lines if 'token' in line.lower())
        
        print(f"  Row operations:     {row_ops}")
        print(f"  Grid operations:    {grid_ops}")
        print(f"  Token operations:   {token_ops}")
        
        # Check for limit_rows
        if 'limit_rows' in func_code:
            # FIX: Extract search result before using in f-string
            match = re.search(r'limit_rows\s*=\s*(\d+)', func_code)
            limit = match.group(1) if match else 'present'
            print(f"  Row limit: {limit}")
        else:
            print("  ⚠️  Row limit not set")
else:
    print("⚠️  arc_loader.py not found")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 32: Candidate Family Analysis (COMPLETE)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 32: Candidate Family Analysis (COMPLETE)")
print("=" * 120)

print("\n📊 Analyzing candidate families (similarity-based clustering):")

family_data = []

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        if len(g) < 3:
            continue
        
        # Cluster by similarity > 90%
        clusters = []
        used = [False] * len(g)
        
        for i in range(len(g)):
            if used[i]:
                continue
            cluster = [i]
            used[i] = True
            a = np.asarray(g[i]["solution"])
            for j in range(i+1, len(g)):
                if used[j]:
                    continue
                b = np.asarray(g[j]["solution"])
                if a.shape == b.shape and np.mean(a == b) * 100 > 90:
                    cluster.append(j)
                    used[j] = True
            if len(cluster) > 1:
                clusters.append(cluster)
        
        if clusters:
            print(f"\n  {bk}:")
            print(f"    Total candidates: {len(g)}")
            print(f"    Families (>90% sim): {len(clusters)}")
            for idx, cluster in enumerate(clusters[:3]):
                print(f"      Family {idx+1}: {len(cluster)} candidates")
                beam_scores = [g[i]["beam_score"] for i in cluster]
                print(f"        Beam scores: {min(beam_scores):.4f} - {max(beam_scores):.4f} (range: {max(beam_scores)-min(beam_scores):.4f})")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 33: Logits Audit (COMPLETE)
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 33: Logits Audit (COMPLETE)")
print("=" * 120)

print("\n📊 Logits Pattern Analysis (inferred from candidate similarity):")

for puzzle_id, subkeys in ALL_BKS.items():
    for bk in subkeys:
        if puzzle_id not in GT:
            continue
        guesses = decoder.decoded_results[bk]
        g = list(guesses.values())
        
        if len(g) < 5:
            continue
        
        # Calculate average similarity between consecutive candidates (proxy for logit smoothness)
        similarities = []
        for i in range(min(10, len(g)-1)):
            a = np.asarray(g[i]["solution"])
            b = np.asarray(g[i+1]["solution"])
            if a.shape == b.shape:
                sim = np.mean(a == b) * 100
                similarities.append(sim)
        
        if similarities:
            avg_sim = np.mean(similarities)
            print(f"\n  {bk}:")
            print(f"    Avg similarity between consecutive: {avg_sim:.2f}%")
            print(f"    Similarity variance: {np.var(similarities):.2f}")
            
            if avg_sim > 90:
                print("    ⚠️  Very high consecutive similarity -> likely logit collapse")
            elif avg_sim > 70:
                print("    ⚠️  Moderate consecutive similarity -> some diversity loss")
            else:
                print("    ✅ Reasonable diversity in consecutive candidates")

print("\n" + "=" * 120)


# ================================================================
# EXPERIMENT 34: Proposed Diversity Penalty Implementation
# ================================================================

print("\n" + "=" * 120)
print("EXPERIMENT 34: Proposed Diversity Penalty Implementation")
print("=" * 120)

print("\n📄 PROPOSED CHANGE TO turbo_dfs:")
print("=" * 80)
print("""
def turbo_dfs_with_diversity(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()
    used_tokens_per_beam = defaultdict(set)  # Track tokens used per beam

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            # Diversity penalty: penalize tokens already used in this beam
            diversity_penalty = 0.0
            if t in used_tokens_per_beam[i]:
                diversity_penalty = 0.5  # Penalize repeated tokens
            # Also penalize tokens that appear in other beams (cross-beam diversity)
            cross_beam_count = sum(1 for j in range(n) if j != i and t in used_tokens_per_beam[j])
            diversity_penalty += 0.1 * cross_beam_count
            
            adjusted_score = score + diversity_penalty
            if adjusted_score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((adjusted_score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((adjusted_score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                used_tokens_per_beam[i].add(t)  # Track token usage
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        # Clone cache to prevent mutation between beams
        import copy
        cache_clone = copy.deepcopy(cache)
        
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache_clone,
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,
            start_time=start_time,
            end_time=end_time,
        )

        del outputs

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes
""")
print("=" * 80)


print("\n" + "=" * 120)
print("✅ EXPERIMENTS 31-34 COMPLETE.")
print("=" * 120)

EXPERIMENT 30: turbo_dfs Line-by-Line Audit

📄 COMPLETE turbo_dfs FUNCTION:
def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
    